# Melody → Chord Transformer | Notebook 01

## Step 1 — Load and inspect one candidate track

**Goal:** Verify that our real training sources have the structure we expect before designing any tensors or building a Transformer.

This step selects a track with **agreement between the two estimated keys** and a **high/medium melody-key confidence**, and checks that the melody, beats, key, and Roman chord JSON files exist. **Agreement is not proof of correctness.** We will measure melody gaps and chord-change alignment in the next step, before filtering tracks for training.

**Run location:** place this notebook in `experiments/pasan/notebooks/` inside your `vocal-harmonization` project. Run the cell below from Jupyter. Only Python standard-library modules are required.


In [1]:
from pathlib import Path
import csv
import json

# Locate the dataset relative to the Jupyter working directory.
# It will work if Jupyter was launched from the project root or a subfolder.
def find_pipeline():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        candidate = folder / "datasets/mtg-jamendo/melody-chord-extraction-pipeline"
        if (candidate / "manifest.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find the pipeline. Open Jupyter from your "
        "vocal-harmonization project folder, then rerun this cell."
    )

pipeline = find_pipeline()
outputs = pipeline / "outputs"

with (pipeline / "manifest.csv").open(newline="", encoding="utf-8") as f:
    manifest = list(csv.DictReader(f))

print(f"Pipeline: {pipeline}")
print(f"Tracks in manifest: {len(manifest):,}")

required = {
    "melody": "melody/combined.json",
    "beats": "beats/beats.json",
    "key": "key.json",
    "roman": "roman/submission.json",
}

# Prefer key-agreement + higher confidence. No arbitrary gap threshold yet.
candidates = []
for row in manifest:
    if row.get("key_agreement", "").strip().lower() != "agree":
        continue
    confidence = row.get("key_confidence", "").strip().lower()
    if confidence not in {"high", "medium"}:
        continue
    track_id = row.get("mtg_track_id", "").strip()
    if not track_id:
        continue
    folder = outputs / track_id
    paths = {name: folder / suffix for name, suffix in required.items()}
    if all(path.is_file() for path in paths.values()):
        candidates.append((confidence, row, paths))

if not candidates:
    raise RuntimeError(
        "No tracks meet the preliminary selection and file checks. "
        "Check beat backfilling and manifest field names."
    )

candidates.sort(key=lambda item: item[0] != "high")
confidence, selected, paths = candidates[0]

print(f"Candidates with agreeing keys and all four files: {len(candidates):,}")
print(f"Selected track: {selected['mtg_track_id']}  (key confidence: {confidence})")
print("Manifest fields:", {
    name: selected.get(name) for name in
    ("key_tonic", "key_mode", "key_agreement", "combined_notes", "tags")
})

# Show actual JSON structure without assuming every file uses the same envelope.
for name, path in paths.items():
    with path.open(encoding="utf-8") as f:
        data = json.load(f)
    print(f"\n{name.upper()}  ({path.relative_to(pipeline)})")
    if isinstance(data, dict):
        print("Top-level keys:", list(data.keys()))
        if name == "key":
            print("Selected key:", data.get("melody_key", {}).get("selected", data.get("selected")))
        else:
            lists = [(k, v) for k, v in data.items() if isinstance(v, list)]
            if lists:
                k, items = lists[0]
                print(f"Array '{k}': {len(items)} records")
                if items:
                    print("First record:", items[0])
            else:
                print("No top-level list found; inspect the keys above.")
    else:
        print("JSON type:", type(data).__name__)
        if isinstance(data, list):
            print("Number of records:", len(data))
            if data:
                print("First record:", data[0])


Pipeline: /Users/praise/Developer/vocal-harmonization/datasets/mtg-jamendo/melody-chord-extraction-pipeline
Tracks in manifest: 1,604
Candidates with agreeing keys and all four files: 203
Selected track: track_0000382  (key confidence: high)
Manifest fields: {'key_tonic': 'C', 'key_mode': 'major', 'key_agreement': 'agree', 'combined_notes': '313', 'tags': 'genre---classical;instrument---voice'}
MELODY  (outputs/track_0000382/melody/combined.json)
Top-level keys: ['schema_version', 'track_id', 'mtg_track_id', 'method', 'source_mp3', 'source_fingerprint', 'time_origin', 'pitch_unit', 'duration_sec', 'notes', 'selection']
Array 'notes': 313 records
First record: {'onset_sec': 0.0099, 'offset_sec': 0.24479, 'midi_pitch': 69, 'midi_pitch_rounded': 69, 'velocity': 100, 'source': 'sheetsage2_instrumental'}
BEATS  (outputs/track_0000382/beats/beats.json)
Top-level keys: ['schema_version', 'track_id', 'mtg_track_id', 'source_mp3', 'source_fingerprint', 'method', 'model_revision', 'time_origin',

### After running this cell

Check whether the four files load and whether the keys, timings, and first records look sensible. Share the printed output. **Next step:** calculate beatwise melody coverage, long gaps, and chords changing within beats for the selected track. Then choose defensible filtering thresholds using the observed data, instead of guessing.


In [2]:
from pathlib import Path
import json

# Select our example track
track_path = Path(
    "/Users/praise/Developer/vocal-harmonization/"
    "datasets/mtg-jamendo/"
    "melody-chord-extraction-pipeline/"
    "outputs/track_0000382"
)

# Load the required JSON files
with open(track_path / "melody/combined.json") as f:
    melody = json.load(f)

with open(track_path / "beats/beats.json") as f:
    beats = json.load(f)

with open(track_path / "roman/submission.json") as f:
    chords = json.load(f)


# -----------------------------------
# 1. Inspect beat information
# -----------------------------------

beat_events = beats["beats"]

print("SONG INFORMATION")
print("----------------------------")
print("Duration:", round(melody["duration_sec"], 2), "seconds")
print("Melody notes:", len(melody["notes"]))
print("Detected beats:", len(beat_events))
print("Chord segments:", len(chords["events"]))

print("\nFIRST 5 BEATS")
for beat in beat_events[:5]:
    print(beat)


# -----------------------------------
# 2. Calculate melody coverage
# -----------------------------------

duration = melody["duration_sec"]

# Sort melody notes by starting time
notes = sorted(
    melody["notes"],
    key=lambda n: n["onset_sec"]
)

# Merge note intervals to avoid counting overlaps twice
intervals = []

for note in notes:
    start = max(0, note["onset_sec"])
    end = min(duration, note["offset_sec"])

    if end <= start:
        continue

    if intervals and start <= intervals[-1][1]:
        intervals[-1][1] = max(intervals[-1][1], end)
    else:
        intervals.append([start, end])

# Total duration containing melody
melody_duration = sum(
    end - start for start, end in intervals
)

coverage = (melody_duration / duration) * 100

print("\nMELODY COVERAGE")
print("----------------------------")
print(f"Melody duration: {melody_duration:.2f} seconds")
print(f"Coverage: {coverage:.2f}%")


# -----------------------------------
# 3. Detect gaps between melody notes
# -----------------------------------

gaps = []

for i in range(len(intervals) - 1):

    gap_start = intervals[i][1]
    gap_end = intervals[i + 1][0]

    gap_duration = gap_end - gap_start

    if gap_duration >= 4.0:
        gaps.append((gap_start, gap_end, gap_duration))

print("\nLARGE INTERNAL MELODY GAPS (>= 4 seconds)")
print("----------------------------")

print("Number of large gaps:", len(gaps))

for start, end, gap in gaps[:10]:
    print(
        f"{start:.2f}s - {end:.2f}s "
        f"(duration: {gap:.2f}s)"
    )


SONG INFORMATION
----------------------------
Duration: 211.04 seconds
Melody notes: 313
Detected beats: 428
Chord segments: 76
FIRST 5 BEATS
{'beat_index': 0, 'time_sec': 0.01, 'beat_in_bar': 1, 'bar_index': 0, 'meter_numerator': 4, 'meter_denominator': 4, 'is_downbeat': True}
{'beat_index': 1, 'time_sec': 0.48, 'beat_in_bar': 2, 'bar_index': 0, 'meter_numerator': 4, 'meter_denominator': 4, 'is_downbeat': False}
{'beat_index': 2, 'time_sec': 1.01, 'beat_in_bar': 3, 'bar_index': 0, 'meter_numerator': 4, 'meter_denominator': 4, 'is_downbeat': False}
{'beat_index': 3, 'time_sec': 1.61, 'beat_in_bar': 4, 'bar_index': 0, 'meter_numerator': 4, 'meter_denominator': 4, 'is_downbeat': False}
{'beat_index': 4, 'time_sec': 2.2, 'beat_in_bar': 1, 'bar_index': 1, 'meter_numerator': 4, 'meter_denominator': 4, 'is_downbeat': True}
MELODY COVERAGE
----------------------------
Melody duration: 162.03 seconds
Coverage: 76.78%
LARGE INTERNAL MELODY GAPS (>= 4 seconds)
----------------------------
Number

In [3]:
import pandas as pd

chord_events = sorted(
    chords["events"],
    key=lambda c: c["start_sec"]
)

rows = []

for i in range(len(beat_events) - 1):

    beat = beat_events[i]

    start = beat["time_sec"]
    end = beat_events[i + 1]["time_sec"]

    beat_duration = end - start

    if beat_duration <= 0:
        continue

    # Calculate overlap between this beat and each chord
    chord_overlap = {}

    for chord in chord_events:

        overlap = max(
            0,
            min(end, chord["end_sec"]) -
            max(start, chord["start_sec"])
        )

        if overlap > 0:

            label = chord["basic_label"]

            chord_overlap[label] = (
                chord_overlap.get(label, 0) + overlap
            )

    # Find chord occupying the most time
    if chord_overlap:

        dominant_chord = max(
            chord_overlap,
            key=chord_overlap.get
        )

        occupancy = (
            chord_overlap[dominant_chord] / beat_duration
        )

        # Count chord labels occupying at least 50 ms
        chord_count = sum(
            t >= 0.05 for t in chord_overlap.values()
        )

    else:
        dominant_chord = None
        occupancy = 0
        chord_count = 0

    rows.append({
        "beat": beat["beat_index"],
        "bar": beat["bar_index"],
        "beat_in_bar": beat["beat_in_bar"],
        "start_sec": start,
        "end_sec": end,
        "chord": dominant_chord,
        "occupancy": occupancy,
        "chord_count": chord_count
    })


beat_chords_df = pd.DataFrame(rows)

print("FIRST 10 BEATS")
display(beat_chords_df.head(10))

print("\nALIGNMENT STATISTICS")
print("----------------------------")

print("Total beat intervals:", len(beat_chords_df))

print(
    "Beats containing multiple chords:",
    (beat_chords_df["chord_count"] > 1).sum()
)

print(
    "Beats with dominant chord below 80% occupancy:",
    (beat_chords_df["occupancy"] < 0.8).sum()
)

print(
    "Beats without a chord label:",
    beat_chords_df["chord"].isna().sum()
)

print("\nEXAMPLES OF MIXED BEATS")

display(
    beat_chords_df[
        beat_chords_df["chord_count"] > 1
    ].head(10)
)


FIRST 10 BEATS


ALIGNMENT STATISTICS
----------------------------
Total beat intervals: 427
Beats containing multiple chords: 58
Beats with dominant chord below 80% occupancy: 41
Beats without a chord label: 0
EXAMPLES OF MIXED BEATS


In [4]:
import json
import numpy as np
import pandas as pd

# Load the estimated musical key
with open(track_path / "key.json") as f:
    key_data = json.load(f)

selected_key = key_data["melody_key"]["selected"]

tonic_pc = selected_key["tonic_pc"]

print("Selected key:", selected_key["tonic"], selected_key["mode"])
print("Tonic pitch class:", tonic_pc)


# -----------------------------------
# Convert melody notes to beat features
# -----------------------------------

melody_rows = []
feature_vectors = []

for _, beat in beat_chords_df.iterrows():

    start = beat["start_sec"]
    end = beat["end_sec"]
    beat_duration = end - start

    # 12 pitch classes: C through B
    pitch_profile = np.zeros(12, dtype=np.float32)

    for note in melody["notes"]:

        note_start = note["onset_sec"]
        note_end = note["offset_sec"]

        # How long does this note overlap the beat?
        overlap = max(
            0,
            min(end, note_end) - max(start, note_start)
        )

        if overlap <= 0:
            continue

        # Convert MIDI pitch to relative pitch class
        midi_pitch = int(note["midi_pitch_rounded"])

        relative_pc = (midi_pitch - tonic_pc) % 12

        # Record the proportion of beat occupied
        pitch_profile[relative_pc] += overlap / beat_duration

    # Total melody activity within this beat
    total_activity = pitch_profile.sum()

    # Melody coverage (assuming monophonic notes)
    melody_coverage = min(float(total_activity), 1.0)

    # Normalize pitch distribution
    if total_activity > 0:
        pitch_distribution = pitch_profile / total_activity
    else:
        pitch_distribution = pitch_profile.copy()

    # 12 pitch features + 1 melody coverage feature
    features = np.append(
        pitch_distribution,
        melody_coverage
    )

    feature_vectors.append(features)

    melody_rows.append({
        "beat": beat["beat"],
        "bar": beat["bar"],
        "beat_in_bar": beat["beat_in_bar"],
        "start_sec": start,
        "end_sec": end,
        "relative_pitches": np.where(
            pitch_profile > 0
        )[0].tolist(),
        "melody_coverage": melody_coverage,
        "chord": beat["chord"],
        "chord_occupancy": beat["occupancy"]
    })


# Create DataFrame
beat_melody_df = pd.DataFrame(melody_rows)

# Convert feature vectors into a NumPy array
X = np.array(feature_vectors, dtype=np.float32)


# -----------------------------------
# Display results
# -----------------------------------

print("\nFIRST 15 BEATS")
display(beat_melody_df.head(15))

print("\nFEATURE ARRAY")
print("----------------------------")
print("Shape:", X.shape)
print("First beat features:", X[0])

print("\nMELODY STATISTICS")
print("----------------------------")

print(
    "Beats without melody:",
    (beat_melody_df["melody_coverage"] == 0).sum()
)

print(
    "Mean beat-level melody coverage:",
    round(beat_melody_df["melody_coverage"].mean() * 100, 2),
    "%"
)


Selected key: C major
Tonic pitch class: 0
FIRST 15 BEATS


FEATURE ARRAY
----------------------------
Shape: (427, 13)
First beat features: [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1.]
MELODY STATISTICS
----------------------------
Beats without melody: 56
Mean beat-level melody coverage: 77.1 %


In [5]:
import numpy as np
import pandas as pd

# -----------------------------------
# 1. Define chord vocabulary
# -----------------------------------

CHORD_FAMILIES = [
    "MAJ",
    "MIN",
    "DOM7",
    "DIM",
    "HDIM7",
    "DIM7",
    "AUG",
    "SUS2",
    "SUS4"
]

NUM_ROOTS = 12
NUM_FAMILIES = len(CHORD_FAMILIES)

NO_CHORD_ID = NUM_ROOTS * NUM_FAMILIES  # 108
NUM_CLASSES = NO_CHORD_ID + 1           # 109

# Used for missing or unsupported labels
IGNORE_ID = -100


# -----------------------------------
# 2. Encode a Roman chord event
# -----------------------------------

def encode_chord(event):

    # No chord
    if event["basic_label"] == "N":
        return NO_CHORD_ID

    root = event["relative_root_semitones"]
    family = event["family"]

    # Unsupported or missing labels
    if root is None or family not in CHORD_FAMILIES:
        return IGNORE_ID

    root_id = int(root)
    family_id = CHORD_FAMILIES.index(family)

    return root_id * NUM_FAMILIES + family_id


# -----------------------------------
# 3. Build label-to-ID mapping
# -----------------------------------

label_to_id = {}

for event in chords["events"]:

    label = event["basic_label"]
    class_id = encode_chord(event)

    if label is None or class_id == IGNORE_ID:
        continue

    # The same chord label must always map
    # to the same numerical ID.
    if label in label_to_id:
        assert label_to_id[label] == class_id

    label_to_id[label] = class_id


# -----------------------------------
# 4. Convert beat-level chords into IDs
# -----------------------------------

y = np.array(
    [
        label_to_id.get(label, IGNORE_ID)
        for label in beat_melody_df["chord"]
    ],
    dtype=np.int64
)


# -----------------------------------
# 5. Verify input-output alignment
# -----------------------------------

assert len(X) == len(y)

assert np.all(
    (y == IGNORE_ID) |
    ((y >= 0) & (y < NUM_CLASSES))
)

print("DATASET SHAPES")
print("----------------------------")
print("Input X:", X.shape)
print("Target y:", y.shape)

print("\nVOCABULARY")
print("----------------------------")
print("Total possible chord classes:", NUM_CLASSES)
print("Unique chord classes in this song:", len(set(y[y >= 0])))

print("\nTARGET STATISTICS")
print("----------------------------")
print("Valid targets:", np.sum(y != IGNORE_ID))
print("Ignored targets:", np.sum(y == IGNORE_ID))
print("No-chord targets:", np.sum(y == NO_CHORD_ID))


# -----------------------------------
# 6. Display aligned examples
# -----------------------------------

preview_df = beat_melody_df[
    [
        "beat",
        "relative_pitches",
        "melody_coverage",
        "chord",
        "chord_occupancy"
    ]
].copy()

preview_df["target_id"] = y

print("\nFIRST 15 ALIGNED BEATS")
display(preview_df.head(15))


# -----------------------------------
# 7. Inspect chord class distribution
# -----------------------------------

print("\nMOST COMMON CHORD LABELS")

display(
    preview_df["chord"]
    .value_counts()
    .head(15)
)


DATASET SHAPES
----------------------------
Input X: (427, 13)
Target y: (427,)
VOCABULARY
----------------------------
Total possible chord classes: 109
Unique chord classes in this song: 17
TARGET STATISTICS
----------------------------
Valid targets: 427
Ignored targets: 0
No-chord targets: 9
FIRST 15 ALIGNED BEATS


MOST COMMON CHORD LABELS


chord
IV:MAJ      110
V:MAJ        89
V:DOM7       52
I:MAJ        38
bVII:MAJ     28
I:DOM7       20
II:MAJ       20
II:DOM7      17
iii:MIN      15
bvii:MIN     11
N             9
ii:MIN        6
i:MIN         4
bii:DIM       3
v:MIN         2
Name: count, dtype: int64


In [6]:
from pathlib import Path
import json
import numpy as np
import pandas as pd


def encode_chord(event):
    """Convert a Roman chord event into a fixed class ID."""

    if event["basic_label"] == "N":
        return NO_CHORD_ID

    root = event.get("relative_root_semitones")
    family = event.get("family")

    if root is None or family not in CHORD_FAMILIES:
        return IGNORE_ID

    root = int(root)

    if not 0 <= root < 12:
        return IGNORE_ID

    return root * len(CHORD_FAMILIES) + CHORD_FAMILIES.index(family)


def preprocess_song(track_path):

    track_path = Path(track_path)

    # ---------------------------------
    # 1. Load song data
    # ---------------------------------

    with open(track_path / "melody/combined.json") as f:
        melody = json.load(f)

    with open(track_path / "beats/beats.json") as f:
        beats = json.load(f)

    with open(track_path / "roman/submission.json") as f:
        chords = json.load(f)

    with open(track_path / "key.json") as f:
        key_data = json.load(f)

    selected_key = key_data["melody_key"]["selected"]

    if selected_key is None:
        raise ValueError("Song has no estimated key")

    tonic_pc = selected_key["tonic_pc"]

    # ---------------------------------
    # 2. Prepare events
    # ---------------------------------

    beat_events = sorted(
        beats["beats"],
        key=lambda b: b["time_sec"]
    )

    melody_notes = sorted(
        melody["notes"],
        key=lambda n: n["onset_sec"]
    )

    chord_events = sorted(
        chords["events"],
        key=lambda c: c["start_sec"]
    )

    X_list = []
    y_list = []
    rows = []

    # ---------------------------------
    # 3. Process each beat
    # ---------------------------------

    for i in range(len(beat_events) - 1):

        beat = beat_events[i]

        start = beat["time_sec"]
        end = beat_events[i + 1]["time_sec"]

        beat_duration = end - start

        if beat_duration <= 0:
            continue

        # -----------------------------
        # A. Encode melody
        # -----------------------------

        pitch_profile = np.zeros(12, dtype=np.float32)

        for note in melody_notes:

            overlap = max(
                0,
                min(end, note["offset_sec"])
                - max(start, note["onset_sec"])
            )

            if overlap <= 0:
                continue

            pitch = int(note["midi_pitch_rounded"])

            relative_pc = (pitch - tonic_pc) % 12

            pitch_profile[relative_pc] += overlap / beat_duration

        total_activity = pitch_profile.sum()

        melody_coverage = min(float(total_activity), 1.0)

        if total_activity > 0:
            pitch_distribution = pitch_profile / total_activity
        else:
            pitch_distribution = pitch_profile.copy()

        features = np.append(
            pitch_distribution,
            melody_coverage
        ).astype(np.float32)

        # -----------------------------
        # B. Encode chords
        # -----------------------------

        chord_overlap = {}
        chord_labels = {}

        for chord in chord_events:

            overlap = max(
                0,
                min(end, chord["end_sec"])
                - max(start, chord["start_sec"])
            )

            if overlap <= 0:
                continue

            chord_id = encode_chord(chord)

            chord_overlap[chord_id] = (
                chord_overlap.get(chord_id, 0) + overlap
            )

            chord_labels[chord_id] = chord["basic_label"]

        if chord_overlap:

            target_id = max(
                chord_overlap,
                key=chord_overlap.get
            )

            chord_occupancy = (
                chord_overlap[target_id] / beat_duration
            )

            chord_count = sum(
                t >= 0.05 for t in chord_overlap.values()
            )

            chord_label = chord_labels[target_id]

        else:

            target_id = IGNORE_ID
            chord_occupancy = 0.0
            chord_count = 0
            chord_label = None

        # -----------------------------
        # C. Save beat information
        # -----------------------------

        X_list.append(features)
        y_list.append(target_id)

        rows.append({
            "beat": beat["beat_index"],
            "bar": beat["bar_index"],
            "beat_in_bar": beat["beat_in_bar"],
            "start_sec": start,
            "end_sec": end,
            "relative_pitches": np.where(
                pitch_profile > 0
            )[0].tolist(),
            "melody_coverage": melody_coverage,
            "chord": chord_label,
            "chord_occupancy": chord_occupancy,
            "chord_count": chord_count,
            "target_id": target_id
        })

    # ---------------------------------
    # 4. Convert to NumPy arrays
    # ---------------------------------

    X = np.array(X_list, dtype=np.float32)

    y = np.array(y_list, dtype=np.int64)

    df = pd.DataFrame(rows)

    # ---------------------------------
    # 5. Return processed song
    # ---------------------------------

    return {
        "track_id": track_path.name,
        "key": selected_key,
        "X": X,
        "y": y,
        "df": df
    }


In [7]:
# Process track 382 using the new function

result = preprocess_song(track_path)

X_new = result["X"]
y_new = result["y"]

print("Track:", result["track_id"])
print("Key:", result["key"]["tonic"], result["key"]["mode"])

print("\nInput shape:", X_new.shape)
print("Target shape:", y_new.shape)

# Compare against the previous preprocessing results

print("\nCOMPARISON WITH PREVIOUS RESULTS")
print("--------------------------------")

print(
    "Melody features match:",
    np.allclose(X, X_new)
)

print(
    "Chord labels match:",
    np.array_equal(y, y_new)
)

print("\nFIRST 10 PROCESSED BEATS")

display(result["df"].head(10))


Track: track_0000382
Key: C major
Input shape: (427, 13)
Target shape: (427,)
COMPARISON WITH PREVIOUS RESULTS
--------------------------------
Melody features match: True
Chord labels match: True
FIRST 10 PROCESSED BEATS


In [8]:
import numpy as np
import json


def preprocess_song_v2(track_path):

    # Run our existing preprocessing function
    result = preprocess_song(track_path)

    X = result["X"]
    df = result["df"]
    key = result["key"]

    # Load beat information
    with open(track_path / "beats/beats.json") as f:
        beats_data = json.load(f)

    # Create a lookup using beat indexes
    beat_lookup = {
        beat["beat_index"]: beat
        for beat in beats_data["beats"]
    }

    # Encode musical mode
    mode = key["mode"]

    if mode not in ("major", "minor"):
        raise ValueError(f"Unsupported mode: {mode}")

    mode_value = 0 if mode == "major" else 1

    extra_features = []

    for row in df.itertuples(index=False):

        beat = beat_lookup[int(row.beat)]

        beat_in_bar = int(beat["beat_in_bar"])
        meter = int(beat["meter_numerator"])

        if meter <= 0 or not (1 <= beat_in_bar <= meter):
            raise ValueError("Invalid beat or meter")

        # Position within the musical bar
        position = (beat_in_bar - 1) / meter

        # Cyclic encoding
        beat_sin = np.sin(2 * np.pi * position)
        beat_cos = np.cos(2 * np.pi * position)

        # Downbeat information
        is_downbeat = float(beat["is_downbeat"])

        extra_features.append([
            beat_sin,
            beat_cos,
            is_downbeat,
            mode_value
        ])

    extra_features = np.array(
        extra_features,
        dtype=np.float32
    )

    # Combine original and new features
    X_updated = np.concatenate(
        [X, extra_features],
        axis=1
    )

    result["X"] = X_updated

    return result


# -----------------------------------
# Test on track 382
# -----------------------------------

result_v2 = preprocess_song_v2(track_path)

X_v2 = result_v2["X"]
y_v2 = result_v2["y"]

print("UPDATED DATASET")
print("--------------------------")
print("Input shape:", X_v2.shape)
print("Target shape:", y_v2.shape)

print("\nFEATURE VERIFICATION")
print("--------------------------")

print(
    "Original 13 features preserved:",
    np.allclose(X_v2[:, :13], X_new)
)

print(
    "Chord targets unchanged:",
    np.array_equal(y_v2, y_new)
)

print("\nFIRST 5 BEATS — NEW FEATURES")

feature_df = pd.DataFrame(
    X_v2[:, 13:],
    columns=[
        "beat_sin",
        "beat_cos",
        "is_downbeat",
        "mode"
    ]
)

display(feature_df.head())


UPDATED DATASET
--------------------------
Input shape: (427, 17)
Target shape: (427,)
FEATURE VERIFICATION
--------------------------
Original 13 features preserved: True
Chord targets unchanged: True
FIRST 5 BEATS — NEW FEATURES


In [9]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Dataset locations
pipeline_path = track_path.parent.parent
outputs_path = pipeline_path / "outputs"

manifest = pd.read_csv(
    pipeline_path / "manifest.csv",
    dtype=str
)

# Initial filtering thresholds
MIN_MELODY_COVERAGE = 0.75
MAX_INTERNAL_GAP = 8.0
MIN_BEATS = 32
MIN_VALID_CHORD_RATIO = 0.95


# --------------------------------------
# Calculate full-song melody statistics
# --------------------------------------

def calculate_melody_stats(notes, duration):

    intervals = []

    for note in notes:

        start = max(0, float(note["onset_sec"]))
        end = min(duration, float(note["offset_sec"]))

        if end > start:
            intervals.append((start, end))

    intervals.sort()

    merged = []

    for start, end in intervals:

        if merged and start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])

    melody_duration = sum(
        end - start for start, end in merged
    )

    coverage = (
        melody_duration / duration if duration > 0 else 0
    )

    internal_gaps = [
        merged[i + 1][0] - merged[i][1]
        for i in range(len(merged) - 1)
    ]

    max_gap = max(internal_gaps, default=0)

    return coverage, max_gap


# --------------------------------------
# Process available tracks
# --------------------------------------

audit_results = []
eligible_tracks = []

required_files = [
    "melody/combined.json",
    "beats/beats.json",
    "roman/submission.json",
    "key.json"
]

for _, track in manifest.iterrows():

    track_id = track["mtg_track_id"]
    song_path = outputs_path / track_id

    record = {
        "track_id": track_id,
        "key_confidence": track["key_confidence"],
        "key_agreement": track["key_agreement"],
        "coverage": np.nan,
        "max_gap": np.nan,
        "beats": np.nan,
        "valid_chord_ratio": np.nan,
        "reason": None
    }

    # 1. Key quality
    if track["key_agreement"] != "agree":
        record["reason"] = "key_disagreement"

    elif track["key_confidence"] not in ("high", "medium"):
        record["reason"] = "low_key_confidence"

    # 2. Required files
    elif not all(
        (song_path / file).exists()
        for file in required_files
    ):
        record["reason"] = "missing_files"

    else:
        try:
            # 3. Run our preprocessing
            result = preprocess_song_v2(song_path)

            X_song = result["X"]
            y_song = result["y"]

            # 4. Calculate melody coverage and gaps
            with open(song_path / "melody/combined.json") as f:
                melody_data = json.load(f)

            coverage, max_gap = calculate_melody_stats(
                melody_data["notes"],
                float(melody_data["duration_sec"])
            )

            # 5. Calculate chord target validity
            valid_ratio = (
                np.mean(y_song != IGNORE_ID)
                if len(y_song) else 0
            )

            record.update({
                "coverage": coverage,
                "max_gap": max_gap,
                "beats": len(X_song),
                "valid_chord_ratio": valid_ratio
            })

            # 6. Apply quality thresholds
            if len(X_song) < MIN_BEATS:
                record["reason"] = "too_few_beats"

            elif coverage < MIN_MELODY_COVERAGE:
                record["reason"] = "low_melody_coverage"

            elif max_gap > MAX_INTERNAL_GAP:
                record["reason"] = "large_melody_gap"

            elif valid_ratio < MIN_VALID_CHORD_RATIO:
                record["reason"] = "invalid_chord_labels"

            else:
                record["reason"] = "eligible"
                eligible_tracks.append(track_id)

        except Exception as e:
            record["reason"] = "processing_error"
            record["error"] = str(e)

    audit_results.append(record)


# --------------------------------------
# Display filtering results
# --------------------------------------

audit_df = pd.DataFrame(audit_results)

print("DATASET QUALITY REPORT")
print("--------------------------------")

print("Total tracks:", len(manifest))
print("Eligible tracks:", len(eligible_tracks))

print("\nFILTERING RESULTS")
display(audit_df["reason"].value_counts())

print("\nELIGIBLE TRACK STATISTICS")

eligible_df = audit_df[
    audit_df["reason"] == "eligible"
]

if not eligible_df.empty:
    display(
        eligible_df[
            ["coverage", "max_gap", "beats",
             "valid_chord_ratio"]
        ].describe()
    )

print("\nFIRST 10 ELIGIBLE TRACKS")
print(eligible_tracks[:10])


DATASET QUALITY REPORT
--------------------------------
Total tracks: 1604
Eligible tracks: 133
FILTERING RESULTS


reason
key_disagreement       1360
eligible                133
low_melody_coverage      67
low_key_confidence       41
large_melody_gap          3
Name: count, dtype: int64


ELIGIBLE TRACK STATISTICS


FIRST 10 ELIGIBLE TRACKS
['track_0000382', 'track_0000386', 'track_0000390', 'track_0000391', 'track_0000394', 'track_0021983', 'track_0021984', 'track_0028194', 'track_0031430', 'track_0031431']


In [10]:
# Check which tracks actually have the required files

audit_check = manifest.copy()

audit_check["has_required_files"] = audit_check["mtg_track_id"].apply(
    lambda track_id: all(
        (outputs_path / track_id / filename).exists()
        for filename in required_files
    )
)

# Handle missing key agreement values
audit_check["key_agreement_clean"] = (
    audit_check["key_agreement"]
    .fillna("not_available")
    .replace("", "not_available")
)

print("DATASET AVAILABILITY")
print("----------------------------")
print("Total tracks:", len(audit_check))

print(
    "Tracks with all required files:",
    audit_check["has_required_files"].sum()
)

print(
    "Tracks missing required files:",
    (~audit_check["has_required_files"]).sum()
)

print("\nKEY AGREEMENT (ALL TRACKS)")
display(
    audit_check["key_agreement_clean"].value_counts()
)

print("\nKEY AGREEMENT (TRACKS WITH ALL FILES)")
display(
    audit_check.loc[
        audit_check["has_required_files"],
        "key_agreement_clean"
    ].value_counts()
)

print("\nEXTRACTION STATUS")
display(
    audit_check["status"]
    .fillna("unknown")
    .value_counts()
)


DATASET AVAILABILITY
----------------------------
Total tracks: 1604
Tracks with all required files: 525
Tracks missing required files: 1079
KEY AGREEMENT (ALL TRACKS)


key_agreement_clean
not_available            1079
disagree                  273
agree                     244
insufficient_evidence       8
Name: count, dtype: int64


KEY AGREEMENT (TRACKS WITH ALL FILES)


key_agreement_clean
disagree                 273
agree                    244
insufficient_evidence      8
Name: count, dtype: int64


EXTRACTION STATUS


status
failed      631
pending     448
partial     416
complete    109
Name: count, dtype: int64


In [11]:
import pandas as pd
from itertools import combinations

# --------------------------------------
# 1. Select eligible tracks
# --------------------------------------

eligible_manifest = manifest[
    manifest["mtg_track_id"].isin(eligible_tracks)
].copy()

print("Total eligible tracks:", len(eligible_manifest))


# --------------------------------------
# 2. Inspect the official split
# --------------------------------------

SPLIT_COLUMN = "split_0"

print("\nORIGINAL SPLIT DISTRIBUTION")
print("--------------------------------")

display(
    eligible_manifest[SPLIT_COLUMN]
    .value_counts(dropna=False)
)


# --------------------------------------
# 3. Standardize split names
# --------------------------------------

eligible_manifest["dataset_split"] = (
    eligible_manifest[SPLIT_COLUMN]
    .str.strip()
    .str.lower()
    .replace({
        "training": "train",
        "valid": "validation",
        "val": "validation",
        "dev": "validation",
        "testing": "test"
    })
)

valid_splits = {"train", "validation", "test"}

unknown_splits = set(
    eligible_manifest["dataset_split"].dropna().unique()
) - valid_splits

if unknown_splits or eligible_manifest["dataset_split"].isna().any():
    raise ValueError(
        f"Missing or unrecognized split labels: {unknown_splits}"
    )


# --------------------------------------
# 4. Create track ID lists
# --------------------------------------

train_tracks = eligible_manifest.loc[
    eligible_manifest["dataset_split"] == "train",
    "mtg_track_id"
].tolist()

val_tracks = eligible_manifest.loc[
    eligible_manifest["dataset_split"] == "validation",
    "mtg_track_id"
].tolist()

test_tracks = eligible_manifest.loc[
    eligible_manifest["dataset_split"] == "test",
    "mtg_track_id"
].tolist()


# --------------------------------------
# 5. Verify track separation
# --------------------------------------

assert len(
    set(train_tracks) &
    set(val_tracks)
) == 0

assert len(
    set(train_tracks) &
    set(test_tracks)
) == 0

assert len(
    set(val_tracks) &
    set(test_tracks)
) == 0

assert (
    len(train_tracks) +
    len(val_tracks) +
    len(test_tracks)
) == len(eligible_manifest)


# --------------------------------------
# 6. Verify artist separation
# --------------------------------------

if eligible_manifest["artist_id"].isna().any():
    raise ValueError("Some eligible tracks have missing artist IDs")

artists_by_split = {
    split: set(
        eligible_manifest.loc[
            eligible_manifest["dataset_split"] == split,
            "artist_id"
        ]
    )
    for split in valid_splits
}

for split_a, split_b in combinations(
    sorted(valid_splits), 2
):
    overlap = (
        artists_by_split[split_a] &
        artists_by_split[split_b]
    )

    print(
        f"Shared artists ({split_a} / {split_b}):",
        len(overlap)
    )

    assert len(overlap) == 0, (
        f"Artist leakage between {split_a} and {split_b}"
    )


# --------------------------------------
# 7. Display results
# --------------------------------------

print("\nFINAL DATASET SPLITS")
print("--------------------------------")

split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Tracks": [
        len(train_tracks),
        len(val_tracks),
        len(test_tracks)
    ]
})

split_summary["Percentage"] = (
    split_summary["Tracks"] /
    len(eligible_manifest) * 100
).round(2)

display(split_summary)

print("\nFIRST 5 TRAINING TRACKS")
print(train_tracks[:5])

print("\nFIRST 5 VALIDATION TRACKS")
print(val_tracks[:5])

print("\nFIRST 5 TEST TRACKS")
print(test_tracks[:5])


Total eligible tracks: 133
ORIGINAL SPLIT DISTRIBUTION
--------------------------------


split_0
train         69
validation    34
test          30
Name: count, dtype: int64


Shared artists (test / train): 0
Shared artists (test / validation): 0
Shared artists (train / validation): 0
FINAL DATASET SPLITS
--------------------------------


FIRST 5 TRAINING TRACKS
['track_0000382', 'track_0000386', 'track_0000390', 'track_0000391', 'track_0000394']
FIRST 5 VALIDATION TRACKS
['track_0046037', 'track_0207464', 'track_0207490', 'track_0207506', 'track_0207545']
FIRST 5 TEST TRACKS
['track_0040620', 'track_0043880', 'track_0043881', 'track_0043883', 'track_0043884']


In [15]:
# Select tracks with disagreeing key estimates
disagreeing = manifest[
    manifest["key_agreement"] == "disagree"
]

# Randomly select 10 tracks
sample = disagreeing.sample(
    n=min(10, len(disagreeing))
)

# Create a readable comparison
comparison = pd.DataFrame({
    "Track": sample["mtg_track_id"],
    "Melody Key": (
        sample["key_tonic"] + " " + sample["key_mode"]
    ),
    "Chord Key": (
        sample["chord_key_tonic"] + " " + sample["chord_key_mode"]
    )
})

display(comparison)


In [16]:
# Convert note names to pitch classes
NOTE_PC = {
    "C": 0, "D": 2, "E": 4,
    "F": 5, "G": 7, "A": 9, "B": 11
}

def tonic_to_pc(tonic):
    tonic = tonic.strip()

    pc = NOTE_PC[tonic[0].upper()]

    for accidental in tonic[1:]:
        if accidental == "#":
            pc += 1
        elif accidental == "b":
            pc -= 1

    return pc % 12


def classify_key_relationship(row):

    m_pc = tonic_to_pc(row["key_tonic"])
    c_pc = tonic_to_pc(row["chord_key_tonic"])

    m_mode = row["key_mode"].lower()
    c_mode = row["chord_key_mode"].lower()

    difference = (c_pc - m_pc) % 12

    # Same tonic and mode, possibly different spelling
    if difference == 0 and m_mode == c_mode:
        return "Same key / enharmonic"

    # Same tonic, different mode
    if difference == 0:
        return "Parallel major/minor"

    # Relative major/minor
    if (
        (m_mode == "major" and c_mode == "minor" and difference == 9)
        or
        (m_mode == "minor" and c_mode == "major" and difference == 3)
    ):
        return "Relative major/minor"

    # Tonics a perfect fourth/fifth apart
    if difference in (5, 7):
        return "Perfect fourth/fifth"

    return "Other"


# Select genuinely disagreeing tracks
disagree_df = manifest[
    manifest["key_agreement"] == "disagree"
].copy()

# Classify each disagreement
disagree_df["relationship"] = disagree_df.apply(
    classify_key_relationship,
    axis=1
)

# Show results
summary = disagree_df["relationship"].value_counts()

print("KEY DISAGREEMENT ANALYSIS")
print("----------------------------")

display(pd.DataFrame({
    "Count": summary,
    "Percentage": (summary / len(disagree_df) * 100).round(2)
}))


KEY DISAGREEMENT ANALYSIS
----------------------------


In [17]:
import numpy as np
import pandas as pd
import json

# --------------------------------------
# Configuration
# --------------------------------------

WINDOW_BARS = 8
BEATS_PER_BAR = 4
WINDOW_LENGTH = WINDOW_BARS * BEATS_PER_BAR

MIN_CHORD_OCCUPANCY = 0.80
MIN_VALID_TARGET_RATIO = 0.75


# --------------------------------------
# Create windows from multiple songs
# --------------------------------------

def create_windows(track_ids):

    all_X = []
    all_y = []
    metadata = []

    skipped_tracks = []
    processing_errors = []

    for track_id in track_ids:

        song_path = outputs_path / track_id

        try:
            # 1. Preprocess song
            result = preprocess_song_v2(song_path)

            X_song = result["X"]
            y_song = result["y"]
            df = result["df"].reset_index(drop=True)

            # 2. Load beat metadata
            with open(song_path / "beats/beats.json") as f:
                beats_data = json.load(f)

            beat_lookup = {
                int(b["beat_index"]): b
                for b in beats_data["beats"]
            }

            # ----------------------------------
            # 3. Organize beats into complete bars
            # ----------------------------------

            bars = []

            for bar_id, group in df.groupby("bar", sort=True):

                indices = group.index.to_numpy()
                beat_ids = group["beat"].astype(int).tolist()

                beat_positions = (
                    group["beat_in_bar"].astype(int).tolist()
                )

                beat_info = [
                    beat_lookup.get(bid)
                    for bid in beat_ids
                ]

                valid_bar = (
                    len(indices) == BEATS_PER_BAR
                    and beat_positions == [1, 2, 3, 4]
                    and np.all(np.diff(indices) == 1)
                    and all(
                        b is not None
                        and int(b["meter_numerator"]) == 4
                        and int(b["meter_denominator"]) == 4
                        for b in beat_info
                    )
                )

                bars.append({
                    "bar_id": int(bar_id),
                    "indices": indices if valid_bar else None
                })

            # ----------------------------------
            # 4. Extract non-overlapping windows
            # ----------------------------------

            windows_created = 0
            cursor = 0

            while cursor <= len(bars) - WINDOW_BARS:

                selected = bars[
                    cursor:cursor + WINDOW_BARS
                ]

                bar_ids = [
                    b["bar_id"]
                    for b in selected
                ]

                # All eight bars must be valid
                valid_bars = all(
                    b["indices"] is not None
                    for b in selected
                )

                # Bar numbers must be consecutive
                consecutive_bars = (
                    bar_ids == list(
                        range(
                            bar_ids[0],
                            bar_ids[0] + WINDOW_BARS
                        )
                    )
                )

                if not (valid_bars and consecutive_bars):
                    cursor += 1
                    continue

                # Combine indices of 8 bars
                indices = np.concatenate([
                    b["indices"]
                    for b in selected
                ])

                # Require consecutive beat records
                if (
                    len(indices) != WINDOW_LENGTH
                    or not np.all(np.diff(indices) == 1)
                ):
                    cursor += 1
                    continue

                # Extract features and targets
                X_window = X_song[indices].copy()
                y_window = y_song[indices].copy()

                # ----------------------------------
                # 5. Mask ambiguous chord labels
                # ----------------------------------

                occupancy = df.iloc[indices][
                    "chord_occupancy"
                ].to_numpy()

                y_window[
                    occupancy < MIN_CHORD_OCCUPANCY
                ] = IGNORE_ID

                valid_targets = int(
                    np.sum(y_window != IGNORE_ID)
                )

                valid_ratio = (
                    valid_targets / WINDOW_LENGTH
                )

                if valid_ratio < MIN_VALID_TARGET_RATIO:
                    cursor += 1
                    continue

                # ----------------------------------
                # 6. Save window
                # ----------------------------------

                all_X.append(X_window)
                all_y.append(y_window)

                metadata.append({
                    "track_id": track_id,
                    "start_bar": bar_ids[0],
                    "end_bar": bar_ids[-1],
                    "valid_targets": valid_targets,
                    "valid_ratio": valid_ratio
                })

                windows_created += 1

                # Move forward by 8 bars
                cursor += WINDOW_BARS

            if windows_created == 0:
                skipped_tracks.append(track_id)

        except Exception as e:
            processing_errors.append({
                "track_id": track_id,
                "error": str(e)
            })

    # --------------------------------------
    # 7. Convert to NumPy arrays
    # --------------------------------------

    if len(all_X) > 0:

        X = np.stack(all_X).astype(np.float32)
        y = np.stack(all_y).astype(np.int64)

    else:

        X = np.empty(
            (0, WINDOW_LENGTH, 17),
            dtype=np.float32
        )

        y = np.empty(
            (0, WINDOW_LENGTH),
            dtype=np.int64
        )

    metadata_df = pd.DataFrame(metadata)

    return X, y, metadata_df, skipped_tracks, processing_errors


# --------------------------------------
# Process each dataset split
# --------------------------------------

train_X, train_y, train_meta, train_skipped, train_errors = (
    create_windows(train_tracks)
)

val_X, val_y, val_meta, val_skipped, val_errors = (
    create_windows(val_tracks)
)

test_X, test_y, test_meta, test_skipped, test_errors = (
    create_windows(test_tracks)
)


# --------------------------------------
# Display results
# --------------------------------------

print("TRAINING WINDOWS")
print("--------------------------------")

print("Input shape:", train_X.shape)
print("Target shape:", train_y.shape)

print("\nVALIDATION WINDOWS")
print("--------------------------------")

print("Input shape:", val_X.shape)
print("Target shape:", val_y.shape)

print("\nTEST WINDOWS")
print("--------------------------------")

print("Input shape:", test_X.shape)
print("Target shape:", test_y.shape)


print("\nDATASET SUMMARY")
print("--------------------------------")

summary = []

for name, X_data, y_data, meta, skipped, errors in [
    ("Train", train_X, train_y, train_meta, train_skipped, train_errors),
    ("Validation", val_X, val_y, val_meta, val_skipped, val_errors),
    ("Test", test_X, test_y, test_meta, test_skipped, test_errors)
]:

    summary.append({
        "Split": name,
        "Windows": len(X_data),
        "Tracks represented": (
            meta["track_id"].nunique()
            if not meta.empty else 0
        ),
        "Tracks without windows": len(skipped),
        "Processing errors": len(errors),
        "Masked target %": (
            round(np.mean(y_data == IGNORE_ID) * 100, 2)
            if y_data.size else 0
        )
    })

display(pd.DataFrame(summary))


print("\nFIRST 5 TRAINING WINDOWS")
display(train_meta.head())


if len(train_X) > 0:

    print("\nFIRST TRAINING EXAMPLE")
    print("--------------------------------")

    print("Input shape:", train_X[0].shape)
    print("Target shape:", train_y[0].shape)

    print("\nFirst 10 chord target IDs:")
    print(train_y[0][:10])

    print("\nMasked targets:")
    print(np.sum(train_y[0] == IGNORE_ID))


print("\nPROCESSING ERRORS")

for name, errors in [
    ("Train", train_errors),
    ("Validation", val_errors),
    ("Test", test_errors)
]:
    print(name, ":", len(errors))

    if errors:
        print("Examples:", errors[:3])


TRAINING WINDOWS
--------------------------------
Input shape: (640, 32, 17)
Target shape: (640, 32)
VALIDATION WINDOWS
--------------------------------
Input shape: (291, 32, 17)
Target shape: (291, 32)
TEST WINDOWS
--------------------------------
Input shape: (361, 32, 17)
Target shape: (361, 32)
DATASET SUMMARY
--------------------------------


FIRST 5 TRAINING WINDOWS


FIRST TRAINING EXAMPLE
--------------------------------
Input shape: (32, 17)
Target shape: (32,)
First 10 chord target IDs:
[ 108   45   45   45   45   45   45   45 -100   90]
Masked targets:
3
PROCESSING ERRORS
Train : 0
Validation : 0
Test : 0


In [18]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------------
# 1. Configuration
# -----------------------------------

BATCH_SIZE = 32

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)


# -----------------------------------
# 2. Convert NumPy arrays to tensors
# -----------------------------------

X_train_tensor = torch.tensor(
    train_X,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    train_y,
    dtype=torch.long
)

X_val_tensor = torch.tensor(
    val_X,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    val_y,
    dtype=torch.long
)

X_test_tensor = torch.tensor(
    test_X,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    test_y,
    dtype=torch.long
)


# -----------------------------------
# 3. Create datasets
# -----------------------------------

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)


# -----------------------------------
# 4. Create DataLoaders
# -----------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# -----------------------------------
# 5. Inspect one training batch
# -----------------------------------

batch_X, batch_y = next(iter(train_loader))

print("\nBATCH INFORMATION")
print("--------------------------------")

print("Input batch shape:", batch_X.shape)
print("Target batch shape:", batch_y.shape)

print("\nInput dtype:", batch_X.dtype)
print("Target dtype:", batch_y.dtype)

print("\nNumber of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of testing batches:", len(test_loader))


# -----------------------------------
# 6. Verify tensor values
# -----------------------------------

print("\nVERIFICATION")
print("--------------------------------")

print(
    "All input values are finite:",
    torch.isfinite(X_train_tensor).all().item()
)

print(
    "Target IDs are valid:",
    (
        (y_train_tensor == IGNORE_ID) |
        (
            (y_train_tensor >= 0) &
            (y_train_tensor < NUM_CLASSES)
        )
    ).all().item()
)

print(
    "Masked targets in this batch:",
    (batch_y == IGNORE_ID).sum().item()
)


PyTorch version: 2.13.0
BATCH INFORMATION
--------------------------------
Input batch shape: torch.Size([32, 32, 17])
Target batch shape: torch.Size([32, 32])
Input dtype: torch.float32
Target dtype: torch.int64
Number of training batches: 20
Number of validation batches: 10
Number of testing batches: 12
VERIFICATION
--------------------------------
All input values are finite: True
Target IDs are valid: True
Masked targets in this batch: 99


In [19]:
import torch
import torch.nn as nn


# -----------------------------------
# 1. Model configuration
# -----------------------------------

INPUT_DIM = 17
D_MODEL = 128


# -----------------------------------
# 2. Input embedding layer
# -----------------------------------

class MelodyEmbedding(nn.Module):

    def __init__(self, input_dim, d_model):

        super().__init__()

        # Convert musical features into
        # learned Transformer embeddings
        self.projection = nn.Linear(
            input_dim,
            d_model
        )

    def forward(self, x):

        # x shape: (batch, sequence, input_dim)

        embedded = self.projection(x)

        # Output: (batch, sequence, d_model)

        return embedded


# -----------------------------------
# 3. Initialize embedding layer
# -----------------------------------

melody_embedding = MelodyEmbedding(
    input_dim=INPUT_DIM,
    d_model=D_MODEL
)


# -----------------------------------
# 4. Test with our actual training batch
# -----------------------------------

batch_X, batch_y = next(iter(train_loader))

embedded_X = melody_embedding(batch_X)


print("INPUT EMBEDDING TEST")
print("--------------------------------")

print("Original input shape:", batch_X.shape)
print("Embedded output shape:", embedded_X.shape)


# -----------------------------------
# 5. Verify output dimensions
# -----------------------------------

assert embedded_X.shape == (
    BATCH_SIZE,
    WINDOW_LENGTH,
    D_MODEL
)

print("\nShape verification: PASSED")


# -----------------------------------
# 6. Inspect learnable parameters
# -----------------------------------

total_parameters = sum(
    p.numel()
    for p in melody_embedding.parameters()
    if p.requires_grad
)

print("\nLEARNABLE PARAMETERS")
print("--------------------------------")

print("Total parameters:", total_parameters)

print("\nWeight matrix shape:")
print(melody_embedding.projection.weight.shape)

print("\nBias vector shape:")
print(melody_embedding.projection.bias.shape)


# -----------------------------------
# 7. Inspect one embedded beat
# -----------------------------------

print("\nFIRST EMBEDDED BEAT")
print("--------------------------------")

print(embedded_X[0, 0, :10].detach().numpy())


INPUT EMBEDDING TEST
--------------------------------
Original input shape: torch.Size([32, 32, 17])
Embedded output shape: torch.Size([32, 32, 128])
Shape verification: PASSED
LEARNABLE PARAMETERS
--------------------------------
Total parameters: 2304
Weight matrix shape:
torch.Size([128, 17])
Bias vector shape:
torch.Size([128])
FIRST EMBEDDED BEAT
--------------------------------
[-0.00514835  0.31740534 -0.3261219  -0.142762   -0.18594582 -0.15406431
 -0.46655816 -0.33534738 -0.27065045 -0.22727117]


In [20]:
import torch
import torch.nn as nn
import math


# -----------------------------------
# 1. Positional Encoding
# -----------------------------------

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=512):

        super().__init__()

        # Create positions:
        # [0, 1, 2, 3, ..., max_len-1]
        position = torch.arange(
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        # Calculate frequencies
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        # Initialize positional encoding matrix
        pe = torch.zeros(max_len, d_model)

        # Even dimensions → sine
        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        # Odd dimensions → cosine
        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        # Add batch dimension
        pe = pe.unsqueeze(0)

        # Store positional encodings in the model
        # without making them trainable parameters
        self.register_buffer("pe", pe)

    def forward(self, x):

        # x: (batch_size, sequence_length, d_model)

        seq_len = x.size(1)

        if seq_len > self.pe.size(1):
            raise ValueError(
                "Sequence exceeds maximum positional length"
            )

        # Add positional information
        x = x + self.pe[:, :seq_len, :]

        return x


# -----------------------------------
# 2. Initialize Positional Encoding
# -----------------------------------

positional_encoding = PositionalEncoding(
    d_model=D_MODEL,
    max_len=512
)


# -----------------------------------
# 3. Apply to embedded melody
# -----------------------------------

positioned_X = positional_encoding(embedded_X)


print("POSITIONAL ENCODING TEST")
print("--------------------------------")

print("Before:", embedded_X.shape)
print("After:", positioned_X.shape)


# -----------------------------------
# 4. Verify dimensions
# -----------------------------------

assert positioned_X.shape == embedded_X.shape

print("\nShape verification: PASSED")


# -----------------------------------
# 5. Inspect positional encodings
# -----------------------------------

print("\nPOSITION 0 — FIRST 10 VALUES")

print(
    positional_encoding.pe[0, 0, :10]
)


print("\nPOSITION 1 — FIRST 10 VALUES")

print(
    positional_encoding.pe[0, 1, :10]
)


# -----------------------------------
# 6. Verify different positions
# -----------------------------------

pos_0 = positional_encoding.pe[0, 0]
pos_1 = positional_encoding.pe[0, 1]

print("\nDifferent positions have different encodings:")

print(
    not torch.allclose(pos_0, pos_1)
)


# -----------------------------------
# 7. Check learnable parameters
# -----------------------------------

parameter_count = sum(
    p.numel()
    for p in positional_encoding.parameters()
)

print("\nLearnable positional encoding parameters:")
print(parameter_count)


POSITIONAL ENCODING TEST
--------------------------------
Before: torch.Size([32, 32, 128])
After: torch.Size([32, 32, 128])
Shape verification: PASSED
POSITION 0 — FIRST 10 VALUES
tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1.])
POSITION 1 — FIRST 10 VALUES
tensor([0.8415, 0.5403, 0.7617, 0.6479, 0.6816, 0.7318, 0.6047, 0.7965, 0.5332,
        0.8460])
Different positions have different encodings:
True
Learnable positional encoding parameters:
0


In [21]:
import torch
import torch.nn as nn
import math


# -----------------------------------
# 1. Multi-Head Self-Attention
# -----------------------------------

class MultiHeadSelfAttention(nn.Module):

    def __init__(self, d_model, num_heads):

        super().__init__()

        assert d_model % num_heads == 0, (
            "d_model must be divisible by num_heads"
        )

        self.d_model = d_model
        self.num_heads = num_heads

        self.head_dim = d_model // num_heads

        # Learnable projections for Q, K, V
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        # Output projection
        self.out_proj = nn.Linear(d_model, d_model)


    def forward(self, x):

        # x: (batch, sequence, d_model)

        batch_size, seq_len, _ = x.shape

        # -----------------------------------
        # A. Calculate Q, K, V
        # -----------------------------------

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        # -----------------------------------
        # B. Split into multiple heads
        # -----------------------------------

        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # Shape:
        # (batch, heads, sequence, head_dim)

        # -----------------------------------
        # C. Calculate attention scores
        # -----------------------------------

        scores = (
            Q @ K.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        # Shape:
        # (batch, heads, seq_len, seq_len)

        # -----------------------------------
        # D. Apply softmax
        # -----------------------------------

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        # -----------------------------------
        # E. Apply attention to values
        # -----------------------------------

        attended = attention_weights @ V

        # Shape:
        # (batch, heads, seq_len, head_dim)

        # -----------------------------------
        # F. Combine attention heads
        # -----------------------------------

        attended = attended.transpose(
            1, 2
        ).contiguous()

        attended = attended.view(
            batch_size,
            seq_len,
            self.d_model
        )

        # -----------------------------------
        # G. Final output projection
        # -----------------------------------

        output = self.out_proj(attended)

        return output, attention_weights


# -----------------------------------
# 2. Initialize attention layer
# -----------------------------------

NUM_HEADS = 4

self_attention = MultiHeadSelfAttention(
    d_model=D_MODEL,
    num_heads=NUM_HEADS
)


# -----------------------------------
# 3. Test with positional embeddings
# -----------------------------------

attention_output, attention_weights = (
    self_attention(positioned_X)
)


print("SELF-ATTENTION TEST")
print("--------------------------------")

print("Input shape:", positioned_X.shape)

print("Attention output shape:", attention_output.shape)

print("Attention weights shape:", attention_weights.shape)


# -----------------------------------
# 4. Verify dimensions
# -----------------------------------

assert attention_output.shape == positioned_X.shape

assert attention_weights.shape == (
    batch_X.shape[0],
    NUM_HEADS,
    WINDOW_LENGTH,
    WINDOW_LENGTH
)

print("\nShape verification: PASSED")


# -----------------------------------
# 5. Verify attention weights
# -----------------------------------

# For each query position, attention
# weights across all key positions sum to 1.

weight_sums = attention_weights.sum(dim=-1)

print("\nAttention weights sum to 1:")

print(
    torch.allclose(
        weight_sums,
        torch.ones_like(weight_sums),
        atol=1e-5
    )
)


# -----------------------------------
# 6. Inspect one attention head
# -----------------------------------

print("\nFIRST HEAD — BEAT 1")

print(
    attention_weights[
        0, 0, 0, :10
    ].detach()
)


# -----------------------------------
# 7. Count learnable parameters
# -----------------------------------

total_parameters = sum(
    p.numel()
    for p in self_attention.parameters()
    if p.requires_grad
)

print("\nLEARNABLE PARAMETERS")
print("--------------------------------")

print("Self-attention parameters:", total_parameters)

print("Number of heads:", NUM_HEADS)

print("Dimensions per head:", self_attention.head_dim)


SELF-ATTENTION TEST
--------------------------------
Input shape: torch.Size([32, 32, 128])
Attention output shape: torch.Size([32, 32, 128])
Attention weights shape: torch.Size([32, 4, 32, 32])
Shape verification: PASSED
Attention weights sum to 1:
True
FIRST HEAD — BEAT 1
tensor([0.0266, 0.0275, 0.0295, 0.0330, 0.0321, 0.0290, 0.0292, 0.0300, 0.0306,
        0.0306])
LEARNABLE PARAMETERS
--------------------------------
Self-attention parameters: 66048
Number of heads: 4
Dimensions per head: 32


In [22]:
import torch
import torch.nn as nn


class TransformerEncoderBlock(nn.Module):

    def __init__(
        self,
        d_model=128,
        num_heads=4,
        d_ff=256,
        dropout=0.1
    ):

        super().__init__()

        # 1. Multi-head self-attention
        self.attention = MultiHeadSelfAttention(
            d_model=d_model,
            num_heads=num_heads
        )

        # 2. Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # 3. Feedforward network
        self.feedforward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

        # 4. Dropout for residual branches
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)


    def forward(self, x):

        # -----------------------------------
        # A. Self-attention + residual
        # -----------------------------------

        normalized = self.norm1(x)

        attention_output, attention_weights = (
            self.attention(normalized)
        )

        x = x + self.dropout1(attention_output)

        # -----------------------------------
        # B. Feedforward + residual
        # -----------------------------------

        normalized = self.norm2(x)

        ff_output = self.feedforward(normalized)

        x = x + self.dropout2(ff_output)

        return x, attention_weights


# -----------------------------------
# Initialize our first encoder block
# -----------------------------------

encoder_block = TransformerEncoderBlock(
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=256,
    dropout=0.1
)


# -----------------------------------
# Test using our melody embeddings
# -----------------------------------

block_output, block_attention = encoder_block(
    positioned_X.detach()
)

print("TRANSFORMER ENCODER BLOCK")
print("--------------------------------")

print("Input:", positioned_X.shape)
print("Output:", block_output.shape)

print("Attention weights:", block_attention.shape)


# -----------------------------------
# Verify dimensions
# -----------------------------------

assert block_output.shape == positioned_X.shape

print("\nShape verification: PASSED")


# -----------------------------------
# Count learnable parameters
# -----------------------------------

total_parameters = sum(
    p.numel()
    for p in encoder_block.parameters()
    if p.requires_grad
)

print("\nLEARNABLE PARAMETERS")
print("--------------------------------")

print("Encoder block parameters:", total_parameters)


# -----------------------------------
# Verify gradient flow
# -----------------------------------

encoder_block.zero_grad()

loss_test = block_output.mean()
loss_test.backward()

gradients_exist = all(
    p.grad is not None
    for p in encoder_block.parameters()
    if p.requires_grad
)

print("\nGRADIENT VERIFICATION")
print("--------------------------------")

print("Gradients computed:", gradients_exist)

print(
    "Output contains finite values:",
    torch.isfinite(block_output).all().item()
)


TRANSFORMER ENCODER BLOCK
--------------------------------
Input: torch.Size([32, 32, 128])
Output: torch.Size([32, 32, 128])
Attention weights: torch.Size([32, 4, 32, 32])
Shape verification: PASSED
LEARNABLE PARAMETERS
--------------------------------
Encoder block parameters: 132480
GRADIENT VERIFICATION
--------------------------------
Gradients computed: True
Output contains finite values: True


In [23]:
import torch
import torch.nn as nn


# -----------------------------------
# 1. Complete Transformer Model
# -----------------------------------

class MelodyChordTransformer(nn.Module):

    def __init__(
        self,
        input_dim=17,
        d_model=128,
        num_heads=4,
        num_layers=3,
        d_ff=256,
        num_classes=109,
        dropout=0.1
    ):

        super().__init__()

        # -----------------------------------
        # A. Melody Embedding
        # -----------------------------------

        self.embedding = MelodyEmbedding(
            input_dim=input_dim,
            d_model=d_model
        )

        # -----------------------------------
        # B. Positional Encoding
        # -----------------------------------

        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            max_len=512
        )

        # -----------------------------------
        # C. Transformer Encoder Layers
        # -----------------------------------

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                num_heads=num_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        # -----------------------------------
        # D. Final Layer Normalization
        # -----------------------------------

        self.final_norm = nn.LayerNorm(d_model)

        # -----------------------------------
        # E. Chord Classification Layer
        # -----------------------------------

        self.output_layer = nn.Linear(
            d_model,
            num_classes
        )


    def forward(self, x):

        # Input shape:
        # (batch, 32, 17)

        # 1. Embed melody features
        x = self.embedding(x)

        # Shape: (batch, 32, 128)

        # 2. Add positional information
        x = self.positional_encoding(x)

        # 3. Pass through Transformer layers
        attention_maps = []

        for layer in self.encoder_layers:

            x, attention_weights = layer(x)

            attention_maps.append(attention_weights)

        # 4. Final normalization
        x = self.final_norm(x)

        # 5. Predict chord-class scores
        logits = self.output_layer(x)

        # Shape: (batch, 32, 109)

        return logits, attention_maps


# -----------------------------------
# 2. Initialize Model
# -----------------------------------

model = MelodyChordTransformer(
    input_dim=INPUT_DIM,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=3,
    d_ff=256,
    num_classes=NUM_CLASSES,
    dropout=0.1
)


# -----------------------------------
# 3. Count Parameters
# -----------------------------------

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("COMPLETE TRANSFORMER MODEL")
print("--------------------------------")

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)


# -----------------------------------
# 4. Test with Actual Training Batch
# -----------------------------------

batch_X, batch_y = next(iter(train_loader))

logits, attention_maps = model(batch_X)

print("\nMODEL OUTPUT")
print("--------------------------------")

print("Input shape:", batch_X.shape)
print("Output shape:", logits.shape)

print("Number of encoder layers:", len(attention_maps))

print(
    "Attention shape (layer 1):",
    attention_maps[0].shape
)


# -----------------------------------
# 5. Verify Model Output
# -----------------------------------

assert logits.shape == (
    batch_X.shape[0],
    WINDOW_LENGTH,
    NUM_CLASSES
)

assert torch.isfinite(logits).all()

print("\nOutput verification: PASSED")


# -----------------------------------
# 6. Test Chord Predictions
# -----------------------------------

predictions = torch.argmax(
    logits,
    dim=-1
)

print("\nPREDICTIONS")
print("--------------------------------")

print("Predictions shape:", predictions.shape)

print("First 10 predicted chord IDs:")
print(predictions[0, :10])

print("\nFirst 10 actual chord IDs:")
print(batch_y[0, :10])


# -----------------------------------
# 7. Test Loss Calculation
# -----------------------------------

loss_fn = nn.CrossEntropyLoss(
    ignore_index=IGNORE_ID
)

loss = loss_fn(
    logits.reshape(-1, NUM_CLASSES),
    batch_y.reshape(-1)
)

print("\nLOSS VERIFICATION")
print("--------------------------------")

print("Initial loss:", loss.item())

assert torch.isfinite(loss)

# -----------------------------------
# 8. Test Backpropagation
# -----------------------------------

model.zero_grad()

loss.backward()

gradients_valid = all(
    p.grad is not None
    and torch.isfinite(p.grad).all().item()
    for p in model.parameters()
    if p.requires_grad
)

print("Gradients valid:", gradients_valid)

assert gradients_valid

print("\nCOMPLETE MODEL VERIFICATION: PASSED")


COMPLETE TRANSFORMER MODEL
--------------------------------
Total parameters: 414061
Trainable parameters: 414061
MODEL OUTPUT
--------------------------------
Input shape: torch.Size([32, 32, 17])
Output shape: torch.Size([32, 32, 109])
Number of encoder layers: 3
Attention shape (layer 1): torch.Size([32, 4, 32, 32])
Output verification: PASSED
PREDICTIONS
--------------------------------
Predictions shape: torch.Size([32, 32])
First 10 predicted chord IDs:
tensor([54, 54, 54, 54, 54, 54, 98, 13, 13, 13])
First 10 actual chord IDs:
tensor([ 1,  1,  1,  1, 72, 72, 72, 72,  1,  1])
LOSS VERIFICATION
--------------------------------
Initial loss: 4.807941913604736
Gradients valid: True
COMPLETE MODEL VERIFICATION: PASSED


In [24]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import copy


# -----------------------------------
# 1. Training configuration
# -----------------------------------

DEVICE = torch.device("cpu")

LEARNING_RATE = 3e-4
MAX_EPOCHS = 30
PATIENCE = 6
GRAD_CLIP = 1.0

model = model.to(DEVICE)


# -----------------------------------
# 2. Loss function and optimizer
# -----------------------------------

# Use sum reduction so we can calculate
# exact average loss across valid beats.

criterion = nn.CrossEntropyLoss(
    ignore_index=IGNORE_ID,
    reduction="sum"
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)


# -----------------------------------
# 3. Function for one epoch
# -----------------------------------

def run_epoch(model, loader, training=True):

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_valid = 0

    for batch_X, batch_y in loader:

        batch_X = batch_X.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        # Identify valid chord targets
        valid_mask = batch_y != IGNORE_ID
        valid_count = valid_mask.sum().item()

        if valid_count == 0:
            continue

        # Reset gradients before training step
        if training:
            optimizer.zero_grad()

        # Enable gradients only during training
        with torch.set_grad_enabled(training):

            # Forward pass
            logits, _ = model(batch_X)

            # Calculate cross-entropy loss
            loss_sum = criterion(
                logits.reshape(-1, NUM_CLASSES),
                batch_y.reshape(-1)
            )

            # Average loss over valid targets
            loss = loss_sum / valid_count

            if training:

                # Backpropagation
                loss.backward()

                # Prevent excessively large gradients
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=GRAD_CLIP
                )

                # Update model parameters
                optimizer.step()

        # -----------------------------------
        # Calculate statistics
        # -----------------------------------

        predictions = logits.argmax(dim=-1)

        correct = (
            (predictions == batch_y) & valid_mask
        ).sum().item()

        total_correct += correct
        total_valid += valid_count
        total_loss += loss_sum.item()

    epoch_loss = total_loss / total_valid
    epoch_accuracy = total_correct / total_valid

    return epoch_loss, epoch_accuracy


# -----------------------------------
# 4. Training loop
# -----------------------------------

history = []

best_val_loss = float("inf")
best_epoch = 0
best_model_state = None

epochs_without_improvement = 0


print("STARTING TRANSFORMER TRAINING")
print("--------------------------------")

for epoch in range(1, MAX_EPOCHS + 1):

    # Train on training dataset
    train_loss, train_accuracy = run_epoch(
        model,
        train_loader,
        training=True
    )

    # Evaluate on validation dataset
    val_loss, val_accuracy = run_epoch(
        model,
        val_loader,
        training=False
    )

    # Record results
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy
    })

    # -----------------------------------
    # Early stopping
    # -----------------------------------

    if val_loss < best_val_loss - 1e-4:

        best_val_loss = val_loss
        best_epoch = epoch

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1

    # Print progress
    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2%} | "
        f"Val Acc: {val_accuracy:.2%}"
    )

    # Stop if validation loss stops improving
    if epochs_without_improvement >= PATIENCE:

        print("\nEarly stopping triggered.")
        break


# -----------------------------------
# 5. Restore best model
# -----------------------------------

model.load_state_dict(best_model_state)
model.eval()


# -----------------------------------
# 6. Display training summary
# -----------------------------------

history_df = pd.DataFrame(history)

print("\nTRAINING COMPLETE")
print("--------------------------------")

print("Epochs completed:", len(history_df))
print("Best epoch:", best_epoch)
print("Best validation loss:", round(best_val_loss, 4))


# -----------------------------------
# 7. Simple majority-class baseline
# -----------------------------------

# Most frequent chord in the training data
valid_train_labels = train_y[
    train_y != IGNORE_ID
]

majority_class = int(
    np.bincount(
        valid_train_labels,
        minlength=NUM_CLASSES
    ).argmax()
)

# Evaluate constant prediction on validation
valid_val_labels = val_y[
    val_y != IGNORE_ID
]

majority_val_accuracy = np.mean(
    valid_val_labels == majority_class
)

print("\nMAJORITY-CLASS BASELINE")
print("--------------------------------")

print("Most common training chord ID:", majority_class)

print(
    "Validation accuracy:",
    f"{majority_val_accuracy:.2%}"
)

print("\nBEST MODEL")
print("--------------------------------")

print(
    "Best validation accuracy:",
    f"{history_df.loc[
        history_df['epoch'] == best_epoch,
        'val_accuracy'
    ].iloc[0]:.2%}"
)


STARTING TRANSFORMER TRAINING
--------------------------------
Epoch 01 | Train Loss: 3.2789 | Val Loss: 2.4270 | Train Acc: 28.56% | Val Acc: 46.13%
Epoch 02 | Train Loss: 2.4825 | Val Loss: 2.0948 | Train Acc: 40.21% | Val Acc: 51.64%
Epoch 03 | Train Loss: 2.2673 | Val Loss: 2.0219 | Train Acc: 40.63% | Val Acc: 51.64%
Epoch 04 | Train Loss: 2.1995 | Val Loss: 2.0077 | Train Acc: 40.63% | Val Acc: 51.64%
Epoch 05 | Train Loss: 2.1676 | Val Loss: 2.0027 | Train Acc: 40.64% | Val Acc: 51.64%
Epoch 06 | Train Loss: 2.1373 | Val Loss: 1.9627 | Train Acc: 40.64% | Val Acc: 51.64%
Epoch 07 | Train Loss: 2.1115 | Val Loss: 2.1059 | Train Acc: 40.57% | Val Acc: 48.59%
Epoch 08 | Train Loss: 2.0932 | Val Loss: 1.9361 | Train Acc: 40.82% | Val Acc: 51.67%
Epoch 09 | Train Loss: 2.0715 | Val Loss: 1.9391 | Train Acc: 40.84% | Val Acc: 51.96%
Epoch 10 | Train Loss: 2.0366 | Val Loss: 1.9731 | Train Acc: 41.25% | Val Acc: 50.20%
Epoch 11 | Train Loss: 2.0031 | Val Loss: 1.9564 | Train Acc: 42.22

In [26]:
import torch
import numpy as np
import pandas as pd

# -----------------------------------
# 1. Convert chord IDs to Roman numerals
# -----------------------------------

ROMAN_ROOTS = [
    "I", "bII", "II", "bIII",
    "III", "IV", "#IV", "V",
    "bVI", "VI", "bVII", "VII"
]


def decode_chord(class_id):

    class_id = int(class_id)

    if class_id == IGNORE_ID:
        return "MASKED"

    if class_id == NO_CHORD_ID:
        return "N"

    if not 0 <= class_id < NO_CHORD_ID:
        return "UNKNOWN"

    root_id = class_id // NUM_FAMILIES
    family_id = class_id % NUM_FAMILIES

    root = ROMAN_ROOTS[root_id]
    family = CHORD_FAMILIES[family_id]

    # Lowercase Roman numerals for minor/diminished
    if family in ("MIN", "DIM", "HDIM7", "DIM7"):

        prefix = ""

        if root.startswith(("b", "#")):
            prefix = root[0]
            root = root[1:]

        root = prefix + root.lower()

    return f"{root}:{family}"


# -----------------------------------
# 2. Collapse consecutive identical chords
# -----------------------------------

def collapse_progression(chords):

    if len(chords) == 0:
        return ""

    progression = []
    current = chords[0]
    count = 1

    for chord in chords[1:]:

        if chord == current:
            count += 1

        else:
            progression.append(
                f"{current} ({count})"
            )

            current = chord
            count = 1

    progression.append(
        f"{current} ({count})"
    )

    return " → ".join(progression)


# -----------------------------------
# 3. Select random validation examples
# -----------------------------------

model.eval()

NUM_EXAMPLES = 3

# Select different songs where possible
unique_tracks = val_meta["track_id"].unique()

selected_tracks = np.random.choice(
    unique_tracks,
    size=min(NUM_EXAMPLES, len(unique_tracks)),
    replace=False
)

selected_indices = []

for track_id in selected_tracks:

    track_indices = np.where(
        val_meta["track_id"].values == track_id
    )[0]

    selected_indices.append(
        np.random.choice(track_indices)
    )


# -----------------------------------
# 4. Generate predictions
# -----------------------------------

examples = []

with torch.no_grad():

    for idx in selected_indices:

        # Get melody window
        X_sample = torch.tensor(
            val_X[idx:idx + 1],
            dtype=torch.float32
        ).to(DEVICE)

        # Actual chord labels
        y_actual = val_y[idx]

        # Predict chords
        logits, _ = model(X_sample)

        y_predicted = (
            logits.argmax(dim=-1)
            .squeeze(0)
            .cpu()
            .numpy()
        )

        # Decode numerical IDs
        actual_chords = [
            decode_chord(c) for c in y_actual
        ]

        predicted_chords = [
            decode_chord(c) for c in y_predicted
        ]

        # Ignore masked positions when calculating accuracy
        valid_mask = y_actual != IGNORE_ID

        accuracy = np.mean(
            y_actual[valid_mask] ==
            y_predicted[valid_mask]
        )

        # Count chord changes between valid adjacent beats
        valid_pairs = valid_mask[:-1] & valid_mask[1:]

        actual_changes = np.sum(
            (y_actual[1:] != y_actual[:-1]) &
            valid_pairs
        )

        predicted_changes = np.sum(
            (y_predicted[1:] != y_predicted[:-1]) &
            valid_pairs
        )

        info = val_meta.iloc[idx]

        examples.append({
            "track_id": info["track_id"],
            "start_bar": info["start_bar"],
            "end_bar": info["end_bar"],
            "accuracy": accuracy,
            "actual": actual_chords,
            "predicted": predicted_chords,
            "actual_changes": int(actual_changes),
            "predicted_changes": int(predicted_changes),
            "valid_beats": int(valid_mask.sum())
        })


# -----------------------------------
# 5. Display progression comparisons
# -----------------------------------

for i, example in enumerate(examples):

    print("\n" + "=" * 65)
    print(f"EXAMPLE {i + 1}")
    print("=" * 65)

    print("Track:", example["track_id"])

    print(
        "Bars:",
        example["start_bar"],
        "-",
        example["end_bar"]
    )

    print(
        "Accuracy:",
        f"{example['accuracy']:.2%}"
    )

    print(
        "Valid beats:",
        example["valid_beats"]
    )

    print("\nACTUAL PROGRESSION")
    print(
        collapse_progression(example["actual"])
    )

    print("\nPREDICTED PROGRESSION")
    print(
        collapse_progression(example["predicted"])
    )

    print("\nCHORD CHANGES")
    print(
        "Actual:",
        example["actual_changes"]
    )

    print(
        "Predicted:",
        example["predicted_changes"]
    )


# -----------------------------------
# 6. Detailed beat-by-beat comparison
# -----------------------------------

print("\n\nDETAILED COMPARISON — EXAMPLE 1")

example = examples[0]

comparison_df = pd.DataFrame({
    "Beat": range(1, WINDOW_LENGTH + 1),
    "Actual": example["actual"],
    "Predicted": example["predicted"]
})

comparison_df["Correct"] = np.where(
    comparison_df["Actual"] == "MASKED",
    "—",
    np.where(
        comparison_df["Actual"] ==
        comparison_df["Predicted"],
        "✓",
        "✗"
    )
)

display(comparison_df)


EXAMPLE 1
Track: track_0571485
Bars: 69 - 76
Accuracy: 77.42%
Valid beats: 31
ACTUAL PROGRESSION
MASKED (1) → bVII:MAJ (2) → i:MIN (4) → bVI:MAJ (2) → v:MIN (1) → i:MIN (21) → N (1)
PREDICTED PROGRESSION
i:MIN (6) → bVI:MAJ (1) → i:MIN (25)
CHORD CHANGES
Actual: 5
Predicted: 2
EXAMPLE 2
Track: track_0915740
Bars: 61 - 68
Accuracy: 35.48%
Valid beats: 31
ACTUAL PROGRESSION
I:MAJ (5) → V:MAJ (8) → MASKED (1) → I:MAJ (15) → IV:MAJ (3)
PREDICTED PROGRESSION
N (1) → I:MAJ (3) → N (1) → I:MAJ (1) → IV:MAJ (5) → I:MAJ (1) → IV:MAJ (7) → I:MAJ (1) → IV:MAJ (2) → I:MAJ (10)
CHORD CHANGES
Actual: 2
Predicted: 9
EXAMPLE 3
Track: track_0854721
Bars: 0 - 7
Accuracy: 43.75%
Valid beats: 32
ACTUAL PROGRESSION
IV:MAJ (4) → I:MAJ (4) → II:SUS4 (4) → I:MAJ (4) → IV:MAJ (4) → I:MAJ (4) → II:SUS4 (4) → I:MAJ (4)
PREDICTED PROGRESSION
I:MAJ (7) → ii:MIN (3) → I:MAJ (12) → IV:MAJ (1) → I:MAJ (9)
CHORD CHANGES
Actual: 7
Predicted: 4
DETAILED COMPARISON — EXAMPLE 1


In [27]:
import torch
import numpy as np
import pandas as pd


# -----------------------------------
# 1. Define harmonic functions
# -----------------------------------

# T  = Tonic
# PD = Predominant / subdominant
# D  = Dominant

MAJOR_FUNCTIONS = {

    # Tonic
    (0, "MAJ"): "T",       # I
    (9, "MIN"): "T",       # vi

    # Predominant
    (5, "MAJ"): "PD",      # IV
    (2, "MIN"): "PD",      # ii

    # Dominant
    (7, "MAJ"): "D",       # V
    (7, "DOM7"): "D",      # V7
    (11, "DIM"): "D",      # vii°
    (11, "DIM7"): "D",
    (11, "HDIM7"): "D"
}


MINOR_FUNCTIONS = {

    # Tonic
    (0, "MIN"): "T",       # i

    # Predominant
    (5, "MIN"): "PD",      # iv
    (2, "DIM"): "PD",      # ii°
    (2, "HDIM7"): "PD",    # iiø7

    # Dominant
    (7, "MAJ"): "D",       # V
    (7, "DOM7"): "D",      # V7
    (11, "DIM"): "D",      # vii°
    (11, "DIM7"): "D"
}


def chord_function(chord_id, mode, extended=False):

    chord_id = int(chord_id)

    # Exclude masked and no-chord positions
    if chord_id < 0 or chord_id >= NO_CHORD_ID:
        return None

    root = chord_id // NUM_FAMILIES
    family = CHORD_FAMILIES[
        chord_id % NUM_FAMILIES
    ]

    if mode == "major":

        function = MAJOR_FUNCTIONS.get(
            (root, family)
        )

        # Extended tonic category:
        # iii can sometimes prolong tonic harmony
        if extended and (root, family) == (4, "MIN"):
            return "T"

    elif mode == "minor":

        function = MINOR_FUNCTIONS.get(
            (root, family)
        )

        # Extended tonic category:
        # bIII may act as a tonic substitute
        if extended and (root, family) == (3, "MAJ"):
            return "T"

    else:
        return None

    return function


# -----------------------------------
# 2. Predict all validation windows
# -----------------------------------

model.eval()

all_predictions = []

with torch.no_grad():

    for batch_X, _ in val_loader:

        batch_X = batch_X.to(DEVICE)

        logits, _ = model(batch_X)

        predictions = logits.argmax(dim=-1)

        all_predictions.append(
            predictions.cpu().numpy()
        )


val_predictions = np.concatenate(
    all_predictions,
    axis=0
)

assert val_predictions.shape == val_y.shape


# -----------------------------------
# 3. Determine musical mode
# -----------------------------------

# Feature 16:
# 0 = Major
# 1 = Minor

val_modes = np.where(
    val_X[:, 0, 16] >= 0.5,
    "minor",
    "major"
)


# -----------------------------------
# 4. Exact, root and family accuracy
# -----------------------------------

# Evaluate actual chord positions only
# (exclude masked and N targets)

valid_mask = (
    (val_y != IGNORE_ID) &
    (val_y != NO_CHORD_ID)
)

predicted_chord_mask = (
    (val_predictions >= 0) &
    (val_predictions < NO_CHORD_ID)
)

total_valid = valid_mask.sum()

exact_matches = (
    (val_y == val_predictions) &
    valid_mask
)

root_matches = (
    (val_y // NUM_FAMILIES ==
     val_predictions // NUM_FAMILIES) &
    valid_mask &
    predicted_chord_mask
)

family_matches = (
    (val_y % NUM_FAMILIES ==
     val_predictions % NUM_FAMILIES) &
    valid_mask &
    predicted_chord_mask
)


print("CHORD PREDICTION METRICS")
print("--------------------------------")

print("Evaluated chord beats:", total_valid)

print(
    "Exact chord accuracy:",
    f"{exact_matches.sum() / total_valid:.2%}"
)

print(
    "Root accuracy:",
    f"{root_matches.sum() / total_valid:.2%}"
)

print(
    "Family accuracy:",
    f"{family_matches.sum() / total_valid:.2%}"
)


# -----------------------------------
# 5. Calculate harmonic-function matches
# -----------------------------------

def evaluate_functions(extended=False):

    actual_functions = np.empty(
        val_y.shape,
        dtype=object
    )

    predicted_functions = np.empty(
        val_y.shape,
        dtype=object
    )

    for i in range(len(val_y)):

        mode = val_modes[i]

        for j in range(val_y.shape[1]):

            actual_functions[i, j] = chord_function(
                val_y[i, j],
                mode,
                extended
            )

            predicted_functions[i, j] = chord_function(
                val_predictions[i, j],
                mode,
                extended
            )

    # Both chords must have recognized functions
    classifiable = (
        valid_mask &
        (actual_functions != None) &
        (predicted_functions != None)
    )

    same_function = (
        classifiable &
        (actual_functions == predicted_functions)
    )

    # Same function but different chord
    functional_substitutions = (
        same_function &
        (val_y != val_predictions)
    )

    return {
        "classifiable": classifiable,
        "same": same_function,
        "substitutions": functional_substitutions,
        "actual": actual_functions,
        "predicted": predicted_functions
    }


strict = evaluate_functions(extended=False)
extended = evaluate_functions(extended=True)


# -----------------------------------
# 6. Display function results
# -----------------------------------

print("\nHARMONIC FUNCTION EVALUATION")
print("--------------------------------")

results = []

for name, result in [
    ("Strict", strict),
    ("Extended", extended)
]:

    classifiable_count = result[
        "classifiable"
    ].sum()

    same_count = result["same"].sum()

    substitution_count = result[
        "substitutions"
    ].sum()

    results.append({
        "Method": name,
        "Coverage %": round(
            classifiable_count / total_valid * 100, 2
        ),
        "Function accuracy (classified) %": round(
            same_count / classifiable_count * 100, 2
        ) if classifiable_count else 0,
        "Function agreement (all chord beats) %": round(
            same_count / total_valid * 100, 2
        ),
        "Different chord, same function": int(
            substitution_count
        )
    })

display(pd.DataFrame(results))


# -----------------------------------
# 7. Inspect functionally similar errors
# -----------------------------------

print("\nEXAMPLES OF FUNCTIONAL SUBSTITUTIONS")
print("--------------------------------")

positions = np.argwhere(
    extended["substitutions"]
)

substitution_examples = []

for window_idx, beat_idx in positions[:15]:

    substitution_examples.append({
        "Track": val_meta.iloc[window_idx]["track_id"],
        "Mode": val_modes[window_idx],
        "Actual": decode_chord(
            val_y[window_idx, beat_idx]
        ),
        "Predicted": decode_chord(
            val_predictions[window_idx, beat_idx]
        ),
        "Function": extended["actual"][
            window_idx, beat_idx
        ]
    })

display(pd.DataFrame(substitution_examples))


# -----------------------------------
# 8. Revisit our previous three examples
# -----------------------------------

print("\nPREVIOUS EXAMPLES — FUNCTION ACCURACY")
print("--------------------------------")

if "selected_indices" in globals():

    for example_number, idx in enumerate(
        selected_indices, start=1
    ):

        valid = valid_mask[idx]

        exact = exact_matches[idx].sum()
        strict_correct = strict["same"][idx].sum()
        extended_correct = extended["same"][idx].sum()

        total = valid.sum()

        print(
            f"Example {example_number} | "
            f"Mode: {val_modes[idx]} | "
            f"Exact: {exact / total:.2%} | "
            f"Strict: {strict_correct / total:.2%} | "
            f"Extended: {extended_correct / total:.2%}"
        )


CHORD PREDICTION METRICS
--------------------------------
Evaluated chord beats: 8295
Exact chord accuracy: 51.90%
Root accuracy: 56.19%
Family accuracy: 73.89%
HARMONIC FUNCTION EVALUATION
--------------------------------


EXAMPLES OF FUNCTIONAL SUBSTITUTIONS
--------------------------------


PREVIOUS EXAMPLES — FUNCTION ACCURACY
--------------------------------
Example 1 | Mode: minor | Exact: 80.00% | Strict: 80.00% | Extended: 80.00%
Example 2 | Mode: major | Exact: 35.48% | Strict: 35.48% | Extended: 35.48%
Example 3 | Mode: major | Exact: 43.75% | Strict: 43.75% | Extended: 43.75%


In [28]:
import numpy as np
import pandas as pd
import torch


# -----------------------------------
# 1. Verify validation data alignment
# -----------------------------------

print("DATA ALIGNMENT CHECK")
print("--------------------------------")

loader_X = []
loader_y = []

for bx, by in val_loader:
    loader_X.append(bx.numpy())
    loader_y.append(by.numpy())

loader_X = np.concatenate(loader_X)
loader_y = np.concatenate(loader_y)

print(
    "Validation features aligned:",
    np.allclose(loader_X, val_X)
)

print(
    "Validation targets aligned:",
    np.array_equal(loader_y, val_y)
)

assert np.allclose(loader_X, val_X)
assert np.array_equal(loader_y, val_y)


# -----------------------------------
# 2. Compare actual vs predicted
#    chord distributions
# -----------------------------------

actual_ids = val_y[valid_mask]
predicted_ids = val_predictions[valid_mask]

actual_counts = (
    pd.Series(actual_ids)
    .value_counts(normalize=True) * 100
)

predicted_counts = (
    pd.Series(predicted_ids)
    .value_counts(normalize=True) * 100
)

# Include commonly occurring actual
# and predicted chord classes
top_ids = list(dict.fromkeys(
    actual_counts.head(8).index.tolist()
    + predicted_counts.head(8).index.tolist()
))

distribution_rows = []

for chord_id in top_ids:

    distribution_rows.append({
        "Chord": decode_chord(chord_id),
        "Actual %": round(
            actual_counts.get(chord_id, 0), 2
        ),
        "Predicted %": round(
            predicted_counts.get(chord_id, 0), 2
        )
    })

distribution_df = pd.DataFrame(distribution_rows)

print("\nCHORD DISTRIBUTION COMPARISON")
print("--------------------------------")

display(distribution_df)

print(
    "\nUnique actual chord classes:",
    len(np.unique(actual_ids))
)

print(
    "Unique predicted chord classes:",
    len(np.unique(predicted_ids))
)


# -----------------------------------
# 3. Analyze chord transitions
# -----------------------------------

# Only compare adjacent beats where
# both actual targets are valid chords.

valid_pairs = (
    valid_mask[:, :-1] &
    valid_mask[:, 1:]
)

actual_changes = (
    val_y[:, 1:] != val_y[:, :-1]
)

predicted_changes = (
    val_predictions[:, 1:] !=
    val_predictions[:, :-1]
)

# True positives:
# Both actual and predicted changed
TP = np.sum(
    actual_changes &
    predicted_changes &
    valid_pairs
)

# False positives:
# Predicted changed, but actual did not
FP = np.sum(
    ~actual_changes &
    predicted_changes &
    valid_pairs
)

# False negatives:
# Actual changed, but predicted did not
FN = np.sum(
    actual_changes &
    ~predicted_changes &
    valid_pairs
)

total_pairs = valid_pairs.sum()

precision = (
    TP / (TP + FP)
    if TP + FP > 0 else 0
)

recall = (
    TP / (TP + FN)
    if TP + FN > 0 else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0 else 0
)


print("\nCHORD CHANGE ANALYSIS")
print("--------------------------------")

print("Evaluated adjacent beat pairs:", total_pairs)

print(
    "Actual chord changes:",
    int(np.sum(actual_changes & valid_pairs))
)

print(
    "Predicted chord changes:",
    int(np.sum(predicted_changes & valid_pairs))
)

print(
    "\nActual change rate:",
    f"{np.mean(actual_changes[valid_pairs]):.2%}"
)

print(
    "Predicted change rate:",
    f"{np.mean(predicted_changes[valid_pairs]):.2%}"
)

print("\nTransition detection metrics")

print("Precision:", f"{precision:.2%}")
print("Recall:", f"{recall:.2%}")
print("F1:", f"{f1:.2%}")


# -----------------------------------
# 4. Function-aware partial credit
# -----------------------------------

# Exact matches are always accepted.
# Different chords receive partial
# recognition if their functions match.

strict_compatible = (
    exact_matches |
    strict["substitutions"]
)

extended_compatible = (
    exact_matches |
    extended["substitutions"]
)

print("\nFUNCTION-AWARE AGREEMENT")
print("--------------------------------")

print(
    "Exact chord accuracy:",
    f"{exact_matches.sum() / total_valid:.2%}"
)

print(
    "Exact or strict functional substitute:",
    f"{strict_compatible.sum() / total_valid:.2%}"
)

print(
    "Exact or extended functional substitute:",
    f"{extended_compatible.sum() / total_valid:.2%}"
)

print(
    "\nStrict functional substitutions:",
    int(strict["substitutions"].sum())
)

print(
    "Extended functional substitutions:",
    int(extended["substitutions"].sum())
)


# -----------------------------------
# 5. Verify previous random examples
# -----------------------------------

print("\nPREVIOUS EXAMPLE VERIFICATION")
print("--------------------------------")

if (
    "selected_indices" in globals()
    and "examples" in globals()
):

    for n, (idx, saved) in enumerate(
        zip(selected_indices, examples),
        start=1
    ):

        current_track = val_meta.iloc[idx]["track_id"]

        valid = val_y[idx] != IGNORE_ID

        current_accuracy = np.mean(
            val_predictions[idx][valid] ==
            val_y[idx][valid]
        )

        saved_predictions = saved["predicted"]

        current_predictions = [
            decode_chord(c)
            for c in val_predictions[idx]
        ]

        print(f"\nExample {n}")

        print("Saved track:", saved["track_id"])
        print("Current track:", current_track)

        print(
            "Saved accuracy:",
            f"{saved['accuracy']:.2%}"
        )

        print(
            "Current accuracy:",
            f"{current_accuracy:.2%}"
        )

        print(
            "Predictions match:",
            saved_predictions == current_predictions
        )

else:
    print("Previous example variables unavailable.")


DATA ALIGNMENT CHECK
--------------------------------
Validation features aligned: True
Validation targets aligned: True
CHORD DISTRIBUTION COMPARISON
--------------------------------


Unique actual chord classes: 29
Unique predicted chord classes: 12
CHORD CHANGE ANALYSIS
--------------------------------
Evaluated adjacent beat pairs: 7383
Actual chord changes: 618
Predicted chord changes: 927
Actual change rate: 8.37%
Predicted change rate: 12.56%
Transition detection metrics
Precision: 10.90%
Recall: 16.34%
F1: 13.07%
FUNCTION-AWARE AGREEMENT
--------------------------------
Exact chord accuracy: 51.90%
Exact or strict functional substitute: 54.07%
Exact or extended functional substitute: 56.50%
Strict functional substitutions: 180
Extended functional substitutions: 382
PREVIOUS EXAMPLE VERIFICATION
--------------------------------
Example 1
Saved track: track_0571485
Current track: track_0571485
Saved accuracy: 77.42%
Current accuracy: 77.42%
Predictions match: True
Example 2
Saved track: track_0915740
Current track: track_0915740
Saved accuracy: 35.48%
Current accuracy: 35.48%
Predictions match: True
Example 3
Saved track: track_0854721
Current track: track_0854

In [29]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd


# -----------------------------------
# 1. Evaluate a set of input windows
# -----------------------------------

def evaluate_melody_input(X_data):

    model.eval()

    all_preds = []
    total_loss = 0.0

    with torch.no_grad():

        for start in range(0, len(X_data), BATCH_SIZE):

            end = start + BATCH_SIZE

            xb = torch.tensor(
                X_data[start:end],
                dtype=torch.float32
            ).to(DEVICE)

            yb = torch.tensor(
                val_y[start:end],
                dtype=torch.long
            ).to(DEVICE)

            logits, _ = model(xb)

            loss = F.cross_entropy(
                logits.reshape(-1, NUM_CLASSES),
                yb.reshape(-1),
                ignore_index=IGNORE_ID,
                reduction="sum"
            )

            total_loss += loss.item()

            predictions = logits.argmax(dim=-1)

            all_preds.append(
                predictions.cpu().numpy()
            )

    predictions = np.concatenate(all_preds)

    # All valid targets, including N
    valid = val_y != IGNORE_ID

    # Chord targets only, excluding N
    chord_valid = valid & (val_y != NO_CHORD_ID)

    overall_accuracy = np.mean(
        predictions[valid] == val_y[valid]
    )

    chord_accuracy = np.mean(
        predictions[chord_valid] == val_y[chord_valid]
    )

    average_loss = total_loss / valid.sum()

    return {
        "Overall accuracy %": overall_accuracy * 100,
        "Chord accuracy %": chord_accuracy * 100,
        "Validation loss": average_loss
    }


# -----------------------------------
# 2. Replace melody with same-mode
#    melody from another song
# -----------------------------------

def replace_melodies(seed):

    rng = np.random.default_rng(seed)

    X_modified = val_X.copy()

    # Major = 0, Minor = 1
    modes = (val_X[:, 0, 16] >= 0.5).astype(int)

    tracks = val_meta["track_id"].to_numpy()

    for i in range(len(val_X)):

        # Find windows with the same mode,
        # but from a different song
        candidates = np.where(
            (modes == modes[i]) &
            (tracks != tracks[i])
        )[0]

        if len(candidates) == 0:
            raise ValueError(
                "No alternative song available "
                f"for mode {modes[i]}"
            )

        donor = rng.choice(candidates)

        # Replace only melody information:
        # Features 0-11: relative pitches
        # Feature 12: melody coverage
        X_modified[i, :, :13] = val_X[
            donor, :, :13
        ]

        # Keep rhythmic position and mode
        # Features 13-16 are unchanged

    return X_modified


# -----------------------------------
# 3. Run original evaluation
# -----------------------------------

results = []

original_result = evaluate_melody_input(val_X)

results.append({
    "Experiment": "Original melody",
    **original_result
})


# -----------------------------------
# 4. Run three random replacements
# -----------------------------------

for seed in [42, 123, 456]:

    X_replaced = replace_melodies(seed)

    result = evaluate_melody_input(X_replaced)

    results.append({
        "Experiment": f"Replaced melody (seed {seed})",
        **result
    })


# -----------------------------------
# 5. Display results
# -----------------------------------

results_df = pd.DataFrame(results)

print("MELODY ABLATION EXPERIMENT")
print("--------------------------------")

display(results_df.round(2))


# -----------------------------------
# 6. Calculate average difference
# -----------------------------------

replaced_results = results_df.iloc[1:]

mean_replaced_accuracy = (
    replaced_results["Chord accuracy %"].mean()
)

original_accuracy = original_result[
    "Chord accuracy %"
]

print("\nMELODY CONTRIBUTION")
print("--------------------------------")

print(
    "Original chord accuracy:",
    f"{original_accuracy:.2f}%"
)

print(
    "Average replaced-melody accuracy:",
    f"{mean_replaced_accuracy:.2f}%"
)

print(
    "Accuracy difference:",
    f"{original_accuracy - mean_replaced_accuracy:+.2f}",
    "percentage points"
)

print(
    "\nOriginal validation loss:",
    round(original_result["Validation loss"], 4)
)

print(
    "Average replaced-melody loss:",
    round(
        replaced_results["Validation loss"].mean(),
        4
    )
)


MELODY ABLATION EXPERIMENT
--------------------------------


MELODY CONTRIBUTION
--------------------------------
Original chord accuracy: 51.90%
Average replaced-melody accuracy: 46.59%
Accuracy difference: +5.31 percentage points
Original validation loss: 1.9257
Average replaced-melody loss: 2.1645


In [30]:
import pandas as pd
import numpy as np
from collections import Counter


# -----------------------------------
# 1. Extract genre labels
# -----------------------------------

def extract_genres(tags):

    if pd.isna(tags):
        return []

    genres = []

    for tag in str(tags).split(";"):

        tag = tag.strip()

        if tag.startswith("genre---"):
            genre = tag.replace("genre---", "", 1)
            genres.append(genre)

    return sorted(set(genres))


# Build track-to-genre mapping
genre_lookup = {}

for _, row in manifest.iterrows():

    track_id = row["mtg_track_id"]

    genre_lookup[track_id] = extract_genres(
        row["tags"]
    )


# -----------------------------------
# 2. Find songs used by the model
# -----------------------------------

used_tracks = sorted(
    train_meta["track_id"].unique()
)

print("TRAINING DATASET")
print("--------------------------------")

print("Training songs:", len(used_tracks))
print("Training windows:", len(train_X))


# -----------------------------------
# 3. Analyze genre distribution
# -----------------------------------

genre_to_tracks = {}

for track_id in used_tracks:

    genres = genre_lookup.get(track_id, [])

    if not genres:
        genres = ["Unlabeled"]

    for genre in genres:

        genre_to_tracks.setdefault(
            genre, []
        ).append(track_id)


genre_rows = []

for genre, tracks in genre_to_tracks.items():

    track_set = set(tracks)

    window_count = (
        train_meta["track_id"].isin(track_set)
    ).sum()

    genre_rows.append({
        "Genre": genre,
        "Songs": len(track_set),
        "Windows": int(window_count),
        "Song %": round(
            len(track_set) / len(used_tracks) * 100,
            2
        )
    })


genre_df = pd.DataFrame(
    genre_rows
).sort_values(
    "Songs",
    ascending=False
).reset_index(drop=True)


print("\nGENRE DISTRIBUTION")
print("--------------------------------")

display(genre_df)


# -----------------------------------
# 4. Analyze major/minor distribution
# -----------------------------------

track_modes = {}

for i, track_id in enumerate(
    train_meta["track_id"]
):

    if track_id not in track_modes:

        mode_value = train_X[i, 0, 16]

        track_modes[track_id] = (
            "minor" if mode_value >= 0.5
            else "major"
        )


mode_rows = []

for genre, tracks in genre_to_tracks.items():

    modes = [
        track_modes[t]
        for t in set(tracks)
    ]

    counts = Counter(modes)

    mode_rows.append({
        "Genre": genre,
        "Major songs": counts["major"],
        "Minor songs": counts["minor"]
    })


print("\nGENRE VS MUSICAL MODE")
print("--------------------------------")

display(
    pd.DataFrame(mode_rows)
    .sort_values(
        "Genre"
    )
    .reset_index(drop=True)
)


# -----------------------------------
# 5. Chord distributions by genre
# -----------------------------------

chord_rows = []

for genre, tracks in genre_to_tracks.items():

    # Find windows belonging to this genre
    indices = np.where(
        train_meta["track_id"].isin(
            set(tracks)
        ).to_numpy()
    )[0]

    genre_targets = train_y[indices].flatten()

    # Remove masked targets and N
    valid_chords = genre_targets[
        (genre_targets >= 0) &
        (genre_targets < NO_CHORD_ID)
    ]

    if len(valid_chords) == 0:
        continue

    chord_counts = Counter(valid_chords)

    # Find five most common chords
    for chord_id, count in chord_counts.most_common(5):

        chord_rows.append({
            "Genre": genre,
            "Chord": decode_chord(chord_id),
            "Beat count": count,
            "Percentage": round(
                count / len(valid_chords) * 100,
                2
            )
        })


chord_genre_df = pd.DataFrame(chord_rows)


print("\nTOP 5 CHORDS PER GENRE")
print("--------------------------------")

display(chord_genre_df)


# -----------------------------------
# 6. Additional dataset statistics
# -----------------------------------

print("\nGENRE LABEL STATISTICS")
print("--------------------------------")

genre_counts_per_song = [
    len(genre_lookup.get(t, []))
    for t in used_tracks
]

print(
    "Songs without genre labels:",
    sum(n == 0 for n in genre_counts_per_song)
)

print(
    "Songs with multiple genre labels:",
    sum(n > 1 for n in genre_counts_per_song)
)

print(
    "Unique genre labels:",
    len(set(
        genre
        for t in used_tracks
        for genre in genre_lookup.get(t, [])
    ))
)


TRAINING DATASET
--------------------------------
Training songs: 52
Training windows: 640
GENRE DISTRIBUTION
--------------------------------


GENRE VS MUSICAL MODE
--------------------------------


TOP 5 CHORDS PER GENRE
--------------------------------


GENRE LABEL STATISTICS
--------------------------------
Songs without genre labels: 0
Songs with multiple genre labels: 41
Unique genre labels: 32


In [31]:
import pandas as pd
import numpy as np
from collections import Counter


# -----------------------------------
# 1. Select genres to investigate
# -----------------------------------

GENRES_TO_COMPARE = [
    "rock",
    "alternative",
    "indie",
    "pop",
    "electronic",
    "classical",
    "jazz"
]

MIN_SONGS = 3


# -----------------------------------
# 2. Analyze genres within each mode
# -----------------------------------

genre_mode_rows = []

for genre in GENRES_TO_COMPARE:

    for mode in ["major", "minor"]:

        # Find training songs matching
        # both genre and musical mode
        matching_tracks = [
            track_id
            for track_id in used_tracks
            if genre in genre_lookup.get(track_id, [])
            and track_modes.get(track_id) == mode
        ]

        # Avoid interpreting very tiny groups
        if len(matching_tracks) < MIN_SONGS:
            continue

        # Find corresponding training windows
        indices = np.where(
            train_meta["track_id"].isin(
                matching_tracks
            ).to_numpy()
        )[0]

        targets = train_y[indices]

        # -----------------------------------
        # Chord distribution
        # -----------------------------------

        valid_chords = targets[
            (targets >= 0) &
            (targets < NO_CHORD_ID)
        ]

        chord_counts = Counter(
            valid_chords.tolist()
        )

        top_chords = chord_counts.most_common(4)

        top_chord_text = " | ".join(
            f"{decode_chord(chord_id)} "
            f"({count / len(valid_chords):.1%})"
            for chord_id, count in top_chords
        )

        # -----------------------------------
        # Chord-change rate
        # -----------------------------------

        valid = (
            (targets >= 0) &
            (targets < NO_CHORD_ID)
        )

        valid_pairs = (
            valid[:, :-1] &
            valid[:, 1:]
        )

        changes = (
            targets[:, 1:] != targets[:, :-1]
        )

        pair_count = valid_pairs.sum()

        change_rate = (
            (changes & valid_pairs).sum()
            / pair_count
            if pair_count > 0
            else np.nan
        )

        genre_mode_rows.append({
            "Genre": genre,
            "Mode": mode,
            "Songs": len(matching_tracks),
            "Windows": len(indices),
            "Chord change rate %": round(
                change_rate * 100, 2
            ),
            "Top 4 chords": top_chord_text
        })


# -----------------------------------
# 3. Display results
# -----------------------------------

genre_mode_df = pd.DataFrame(
    genre_mode_rows
)

print("GENRE COMPARISON WITHIN MUSICAL MODE")
print("--------------------------------")

display(genre_mode_df)


# -----------------------------------
# 4. Check overlap between genre groups
# -----------------------------------

print("\nGENRE OVERLAP")
print("--------------------------------")

overlap_rows = []

for i, genre_a in enumerate(GENRES_TO_COMPARE):

    for genre_b in GENRES_TO_COMPARE[i + 1:]:

        tracks_a = set(
            genre_to_tracks.get(genre_a, [])
        )

        tracks_b = set(
            genre_to_tracks.get(genre_b, [])
        )

        overlap = tracks_a & tracks_b

        if len(overlap) > 0:

            overlap_rows.append({
                "Genre A": genre_a,
                "Genre B": genre_b,
                "Shared songs": len(overlap)
            })


overlap_df = pd.DataFrame(
    overlap_rows
).sort_values(
    "Shared songs",
    ascending=False
)

display(overlap_df.head(15))


GENRE COMPARISON WITHIN MUSICAL MODE
--------------------------------


GENRE OVERLAP
--------------------------------
